Note: This notebook contains a version of the code used in our work, provided for reproducibility. 

Some of the components have been removed or modified for confidentiality. As part of this cleanup, the code has been refactored for readability, and may differ slightly from the exact execution environment used in the original experiments.

That said, the main logic remain unchanged and represents the approach described in the paper.

# Generate LLM Feedback

## Import

In [ ]:
import sys
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizer, PreTrainedModel
import transformers
#from transformers.modeling_utils import PreTrainedModel
#from transformers.tokenization_utils import PreTrainedTokenizer
from huggingface_hub import login
import matplotlib.pyplot as plt
import glob
import json
import numpy as np
import time
import random
import os
import copy
from pathlib import Path
import re
from typing import List

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
login(token=os.environ["HUGGINGFACE_TOKEN"])

## Pipeline

To generate feedback for every paragraph (with at least one GT) of every file.

### Initiate Model

Choose the model you want to use to generate feedback

#### Llama

In [ ]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    token=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    use_auth_token=True,
    trust_remote_code=True
)
#model = model.to("cuda")
model.eval()

#### DeepSeek

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")

model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    device_map="auto",               
    dtype=torch.float16
)


model.eval()

#### Mistral

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    use_auth_token=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    use_auth_token=True,
    trust_remote_code=True
)
#model = model.to("cuda")
model.eval()

#### QWEN 72B

In [ ]:
model_name = "Qwen/Qwen2.5-72B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    use_auth_token=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    use_auth_token=True,
    trust_remote_code=True
)
#model = model.to("cuda")
model.eval()

### Paragraph Level

#### Zero-Shot

##### Unguided Prompts

In [ ]:
PROMPT_TEMPLATE = (
"""
You are a professional writing instructor providing very concise, useful feedback. You should spot obvious errors and mention surface-level improvements, but your main focus is to offer incisive, deep, insightful, and actionable advice that helps the student improve. When a passage is especially strong and offers little room for improvement, acknowledge that briefly. Your response should be concise, to the point, and limited to a few sentences, each addressing a specific improvement or commendation. While keeping the full essay in mind, focus only on the target paragraph. If something is wrong with a specific part, sentence, or word, explicitly identify that part.

### Essay:\n{context}
### Target Paragraph:\n{paragraph}
### Provide feedback to the student on the target paragraph:\n
"""
)

##### Guided Prompts

In [ ]:
PROMPT_TEMPLATE = (
"""
You are a professional writing instructor providing very concise, useful feedback. You should spot obvious errors and mention surface-level improvements, but your main focus is to offer incisive, deep, insightful, and actionable advice that helps the student improve. When a passage is especially strong and offers little room for improvement, acknowledge that briefly. Your response should be concise, to the point, and limited to a few sentences, each addressing a specific improvement or commendation. While keeping the full essay in mind, focus only on the target paragraph. If something is wrong with a specific part, sentence, or word, explicitly identify that part.

**Feedback categories/options:**
    *Task Constraints* - Clarifies goals, requirements, or rubric expectations.
    *Concepts* - Defines or explains key terms or conceptual context.
    *Elaboration* - Encourages expansion, depth, or development of ideas/scenes.
    *Clarification* - Flags confusion and asks for clearer meaning or context.
    *Mistakes* - Identifies and explains errors (grammar, logic, organization, repetition).
    *Praise* - Pure encouragement or positive reaction only.
    *None* - Personal opinion, symbols only, or mixed feedback types.

### Essay:\n{context}
### Target Paragraph:\n{paragraph}
### Provide feedback to the student on the target paragraph:\n
"""
)


#### Few-Shot

###### Unguided

In [ ]:
# Just the selected samples to be included in the prompt.

# Samples removed for privacy. Insert your selected samples here.
# "[Click to expand]" is used to indicate that the full essay is not shown in the prompt, to provide more context regarding the target essay and paragraph within the token limit.

few_shot_samples = ("""
## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 1 PARAGRAPH
### Instructor's feedback regarding the paragraph:
SAMPLE 1 FEEDBACK  

## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 2 PARAGRAPH
### Instructor's feedback regarding the paragraph:
SAMPLE 2 FEEDBACK  
    
## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 3 PARAGRAPH
### Instructor's feedback regarding the paragraph:
SAMPLE 3 FEEDBACK  

## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 4 PARAGRAPH
### Instructor's feedback regarding the paragraph:
SAMPLE 4 FEEDBACK  

## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 5 PARAGRAPH
### Instructor's feedback regarding the paragraph:
SAMPLE 5 FEEDBACK  

""")

In [ ]:
PROMPT_TEMPLATE = (  # Unguided
"""
You are a professional writing instructor providing very concise, useful feedback. You should spot obvious errors and mention surface-level improvements, but your main focus is to offer incisive, deep, insightful, and actionable advice that helps the student improve. When a passage is especially strong and offers little room for improvement, acknowledge that briefly. Your response should be concise, to the point, and limited to a few sentences, each addressing a specific improvement or commendation. While keeping the full essay in mind, focus only on the target paragraph. If something is wrong with a specific part, sentence, or word, explicitly identify that part.


{samples}

## Student:
### Essay:
{{context}}
### Paragraph:
{{paragraph}}
### Instructor's feedback regarding the paragraph:
""".format(samples=few_shot_samples)
)

##### Guided Generic

In [ ]:
# Just the selected samples to be included in the prompt

# Samples removed for privacy. Insert your selected samples here.
# "[Click to expand]" is used to indicate that the full essay is not shown in the prompt, to provide more context regarding the target essay and paragraph within the token limit.

guided_samples=(
"""
## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 1 PARAGRAPH
### Instructor's feedback regarding the paragraph:
1) SAMPLE 1 FEEDBACK 1
2) SAMPLE 1 FEEDBACK 2
  

## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 2 PARAGRAPH
### Instructor's feedback regarding the paragraph:
1) SAMPLE 2 FEEDBACK 1
2) SAMPLE 2 FEEDBACK 2
3) SAMPLE 2 FEEDBACK 3


    
## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 3 PARAGRAPH
### Instructor's feedback regarding the paragraph:
1) SAMPLE 3 FEEDBACK 1
2) SAMPLE 3 FEEDBACK 2
3) SAMPLE 3 FEEDBACK 3


## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 4 PARAGRAPH
### Instructor's feedback regarding the paragraph:
1) SAMPLE 4 FEEDBACK 1


## Student:
### Essay: [Click to expand]
### Paragraph:
SAMPLE 5 PARAGRAPH
### Instructor's feedback regarding the paragraph:
1) SAMPLE 5 FEEDBACK 1
2) SAMPLE 5 FEEDBACK 2
3) SAMPLE 5 FEEDBACK 3
""")

In [ ]:
PROMPT_TEMPLATE = (  # Guided. Generic Prompt
"""
You are a professional writing instructor providing very concise, useful feedback. You should spot obvious errors and mention surface-level improvements, but your main focus is to offer incisive, deep, insightful, and actionable advice that helps the student improve. When a passage is especially strong and offers little room for improvement, acknowledge that briefly. Your response should be concise, to the point, and limited to a few sentences, each addressing a specific improvement or commendation. While keeping the full essay in mind, focus only on the target paragraph. If something is wrong with a specific part, sentence, or word, explicitly identify that part.

**Feedback categories/options:**
    *Task Constraints* - Clarifies goals, requirements, or rubric expectations.
    *Concepts* - Defines or explains key terms or conceptual context.
    *Elaboration* - Encourages expansion, depth, or development of ideas/scenes.
    *Clarification* - Flags confusion and asks for clearer meaning or context.
    *Mistakes* - Identifies and explains errors (grammar, logic, organization, repetition).
    *Praise* - Pure encouragement or positive reaction only.
    *None* - Personal opinion, symbols only, or mixed feedback types.


{samples}

## Student:
### Essay: 
{{context}}
### Paragraph:
{{paragraph}}
### Instructor's feedback regarding the paragraph:
""".format(samples=guided_samples)
)


#### Experiment Settings

In [ ]:
OUTPUT_PREFIX = ''
MAX_CONTEXT_LENGTH = model.config.max_position_embeddings


import os
from typing import Optional

def fetch_rubric_from_filename(
    filename: str,
    dataset_root: str = "./SEFORA"
) -> Optional[str]:
    """
    Given a filename in formal:
    Batch_#_Course_#_Class_#_Essay_#_Initial_ID####.json

    Walks from the deepest inferred directory upward and
    returns the contents of the first .txt file found.
    """

    name = os.path.basename(filename)
    if not name.endswith(".json"):
        raise ValueError(f"Not a JSON file: {filename}")

    parts = name[:-5].split("_")  # remove .json, split
    dataset_root = os.path.abspath(dataset_root)

    # Parse filename (two supported formats)
    if "Class" in parts:
        # Batch_#, Course_#, Class_#, Essay_#, STAGE, ID#
        try:
            batch  = "_".join(parts[0:2])   # Batch_#
            course = "_".join(parts[2:4])   # Course_#
            clazz  = "_".join(parts[4:6])   # Class_#
            essay  = "_".join(parts[6:8])   # Essay_#
            stage  = parts[8]               # STAGE
            path_parts = [dataset_root, batch, course, clazz, essay, stage]
        except IndexError:
            print(f"Unexpected filename format: {filename} RETURNING EMPTY")
            return ""
    else:
        # Batch_#, Course_#, Essay_#, STAGE, ID# , if no class, then  class is always 1
        try:
            batch  = "_".join(parts[0:2])
            course = "_".join(parts[2:4])
            clazz  = "Class_1"
            essay  = "_".join(parts[4:6])
            stage  = parts[6]
            path_parts = [dataset_root, batch, course, clazz, essay, stage]
        except IndexError:
            print(f"Unexpected filename format: {filename} RETURNING EMPTY")
            return ""
            #raise ValueError(f"Unexpected filename format: {filename}")

    # Start from deepest directory
    current_dir = os.path.abspath(os.path.join(*path_parts))
    while not os.path.isdir(current_dir) and current_dir != dataset_root:
        current_dir = os.path.dirname(current_dir)


    # Walk upward until dataset_root
    while True:
        if os.path.isdir(current_dir):
            for fname in os.listdir(current_dir):
                if fname.lower().endswith(".txt"):
                    rubric_path = os.path.join(current_dir, fname)
                    with open(rubric_path, "r", encoding="utf-8") as f:
                        return f.read()

        if current_dir == dataset_root:
            break

        parent = os.path.dirname(current_dir)
        if parent == current_dir:
            break

        current_dir = parent

    return ""

def maybe_wrap_for_qwen(prompt: str, tokenizer: PreTrainedTokenizer) -> str:
    """
    Wrap prompt in a chat template if the tokenizer supports it (Qwen).
    Otherwise, return unchanged.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "You are a writing instructor."},
            {"role": "user", "content": prompt},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    return prompt

def remove_half_sentence(text: str) -> str:
    """
    Remove the last incomplete sentence from the text if it doesn't end
    with a sentence terminator (., !, ?, ;) optionally followed by quotes or brackets.
    If no terminator is found anywhere, returns the original text.
    """
    if not text:
        return text

    # Strip trailing whitespace for analysis
    stripped = text.rstrip()

    # Check if the text ends with a terminator (., !, ?, ;) possibly followed by quotes/brackets
    if re.search(r'[\.!?;][\'"\)\]\}]*\Z', stripped):
        return text  # Already ends with a complete sentence

    # Find the last position of any sentence terminator
    last_pos = max(
        stripped.rfind('.'),
        stripped.rfind('!'),
        stripped.rfind('?'),
        stripped.rfind(';')
    )
    if last_pos == -1:
        return text  # No terminator at all

    # Truncate text right after the terminator
    truncated = stripped[:last_pos + 1]
    return truncated
    

def make_prompt(paragraphs: list, idx: int, tokenizer: PreTrainedTokenizer, max_tokens: int, prompt_overhead: int, rubric: str) -> str:
    """
    Build a prompt for paragraph at index idx, including preceding and following context paragraphs
    truncated to fit within the model's context window by removing the largest-side paragraphs first.
    """
    target = paragraphs[idx]

    # Build initial context list: all paragraphs including target
    all_ctx = []
    # include preceding paragraphs (in order)
    for j in range(idx + 1):
        all_ctx.append(paragraphs[j])
    # include following paragraphs
    for j in range(idx + 1, len(paragraphs)):
        all_ctx.append(paragraphs[j])

    # Helper to count words in a list of paragraphs
    def word_count(paras):
        return sum(len(p.split()) for p in paras)

    # Trim paragraphs until the token limit is satisfied
    while True:
        # Generate the context string
        context = "\n".join(all_ctx).strip()
        # Compute token lengths
        ctx_len = tokenizer(context, return_tensors="pt")["input_ids"].size(1)
        tgt_len = tokenizer(target, return_tensors="pt")["input_ids"].size(1)
        #print("ctx_len:", ctx_len, "tgt_len", tgt_len)
        # Check if within limits (+20 safety buffer)
        
        CONTEXT_BUDGET = int(0.36 * max_tokens)

        if ctx_len + prompt_overhead <= CONTEXT_BUDGET:
            break 
        #if ctx_len + tgt_len + prompt_overhead + GEN_PARAMS["max_new_tokens"] + 20 <= max_tokens: # tries to use THE WHOLE CONTEXT WINDOW
        #    break
        # If only the target remains, stop trimming
        if len(all_ctx) == 0:
            break
        # Determine words before and after the target in all_ctx
        tgt_pos = all_ctx.index(target)
        before = word_count(all_ctx[:tgt_pos])
        after = word_count(all_ctx[tgt_pos+1:])
        # Remove from the side with more words; never remove the target
        if before > after and tgt_pos > 0:
            all_ctx.pop(0)
        elif len(all_ctx) - 1 > tgt_pos:
            all_ctx.pop()
        else:
            # Fallback: remove from beginning if no other choice
            all_ctx.pop(0)

    # Final context and prompt formatting
    final_context = "\n".join(all_ctx).strip()
    if "{rubric}" in PROMPT_TEMPLATE:
        return PROMPT_TEMPLATE.format(context=final_context, paragraph=target, rubric=rubric)
        
    return PROMPT_TEMPLATE.format(context=final_context, paragraph=target)



def process_json_files(json_dir: str,
                       num_files: int = None,
                       continue_ind: int = None,
                       batch_size: int = 8,
                       seed: int = 42,
                       setting: str = "default",
                       save_dir: str = None,
                       tokenizer: PreTrainedTokenizer = None,
                       model: PreTrainedModel = None,
                       use_rubric: bool = False,
                       sub56: bool = False) -> list:
    """
    Sample files and generate feedback.
    """
    # collect filepaths
    json_dir = os.path.expanduser(json_dir)    
    paths = glob.glob(os.path.join(json_dir, '**', '*.json'), recursive=True)
    if num_files is not None:
        random.seed(seed)
        paths = random.sample(paths, min(num_files, len(paths)))
    # delegate
    #print(paths)
    #return
    if continue_ind is None:
        continue_ind = 0
    return run_exp(paths, continue_ind, tokenizer, model, setting, batch_size, seed, save_dir, use_rubric, sub56)

In [ ]:
GEN_PARAMS = {
    "max_new_tokens": 250, # For faster reference and testing
    "temperature": 0.1,
    "top_p": 1,
    "repetition_penalty": 1.2,
} 


def run_exp(filepaths: list,
            continue_ind: int,
            tokenizer: PreTrainedTokenizer,
            model: PreTrainedModel,
            setting: str = "default",
            batch_size: int = 8,
            seed: int = 42,
            save_dir: str = None,
            use_rubric: bool = False,
            sub56: bool = False) -> list:
    """
    Process a list of JSON files, generating feedback for each paragraph in parallel batches.
    """

    # Verify pad_token and padding_side
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(tokenizer, 'padding_side', None) != 'left':
        tokenizer.padding_side = 'left'

    # Determine context window and prompt overhead
    max_tokens = MAX_CONTEXT_LENGTH
    empty_prompt = ""
    if use_rubric:
        empty_prompt = PROMPT_TEMPLATE.format(context="", paragraph="", rubric="")
    else:
        empty_prompt = PROMPT_TEMPLATE.format(context="", paragraph="")
    prompt_overhead = tokenizer(empty_prompt, return_tensors="pt")["input_ids"].size(1)

    random.seed(seed)
    results = []
    
    sub56_path = "sub56.txt"
    if sub56:
        if not os.path.isfile(sub56_path):
            raise FileNotFoundError(f"{sub56_path} not found")
        with open(sub56_path, "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}


    cnt = 0
    for filepath in filepaths:
        if cnt < continue_ind:
            cnt += 1
            continue

        cnt += 1
        print("Processing ", cnt, '/', len(filepaths), filepath)
        if sub56 and not (filepath.split('/')[-1] in sub56_names):
            print("NOT SELECTED... SKIPPED...")
            continue
            
        os.system("nvidia-smi > ~/logs/GPU_Stat_" + setting)

        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        original_paragraphs = data.get('paragraphs', [])

        rubric = ""
        if use_rubric:
            rubric = fetch_rubric_from_filename(filepath)
            
        # FULL bodies list — used ONLY for context
        bodies_all = [
            p.get('body', '').strip() or p.get('text', '').strip()
            for p in original_paragraphs
        ]

        # indices of paragraphs that have annotations (generation targets)
        target_indices = [
            i for i, p in enumerate(original_paragraphs)
            if p.get("annotations")
        ]

        # deep copy ENTIRE JSON so everything stays identical
        out_entry = json.loads(json.dumps(data))

        # nothing to generate
        if not target_indices:
            results.append(out_entry)
            if save_dir:
                save_dir = os.path.expanduser(save_dir)
                os.makedirs(save_dir, exist_ok=True)
                base = os.path.basename(filepath)
                out_path = os.path.join(save_dir, f"{base}")
                with open(out_path, 'w', encoding='utf-8') as wf:
                    json.dump(out_entry, wf, ensure_ascii=False, indent=2)
            continue

        # Process in batches over TARGET indices
        for start in range(0, len(target_indices), batch_size):
            batch_orig_indices = target_indices[start:start + batch_size]

            # Build prompts USING FULL CONTEXT
            #prompts = [
            #    make_prompt(
            #        bodies_all,
            #        orig_idx,
            #        tokenizer,
            #        max_tokens,
            #        prompt_overhead,
            #        rubric
            #    )
            #    for orig_idx in batch_orig_indices
            #]
            prompts = [ # So it handles QWEN 72B prompt templates
                maybe_wrap_for_qwen(
                    make_prompt(
                        bodies_all,
                        orig_idx,
                        tokenizer,
                        max_tokens,
                        prompt_overhead,
                        rubric
                    ),
                    tokenizer
                )
                for orig_idx in batch_orig_indices
            ]


            #print(prompts[0])

            # Tokenize
            batch_inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True
            )
            batch_inputs = {k: v.to(model.device) for k, v in batch_inputs.items()}

            input_size = batch_inputs['input_ids'].shape[1]

            if seed is not None:
                torch.manual_seed(seed)

            with torch.no_grad():
                output_ids = model.generate(
                    **batch_inputs,
                    **GEN_PARAMS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Decode and WRITE BACK IN PLACE
            for i, orig_idx in enumerate(batch_orig_indices):
                seq = output_ids[i]
                feedback_tokens = seq[input_size:]
                feedback = tokenizer.decode(
                    feedback_tokens,
                    skip_special_tokens=True
                ).strip()

                feedback = remove_half_sentence(feedback)

                entry = out_entry['paragraphs'][orig_idx]

                if 'model' not in entry:
                    entry['model'] = []

                for model_entry in entry['model']:
                    if model_entry.get('setting') == setting:
                        model_entry.setdefault('feedback', []).append(feedback)
                        break
                else:
                    entry['model'].append({
                        'setting': setting,
                        'feedback': [feedback]
                    })

        results.append(out_entry)

        # save if needed
        if save_dir:
            save_dir = os.path.expanduser(save_dir)
            os.makedirs(save_dir, exist_ok=True)
            base = os.path.basename(filepath)
            out_path = os.path.join(save_dir, f"{base}")
            with open(out_path, 'w', encoding='utf-8') as wf:
                json.dump(out_entry, wf, ensure_ascii=False, indent=2)

    return results


In [ ]:
GPT_SIM_PROMPT = (
    """You are to provide a similarity score in the scale of 0 to 4, as to how similar the two feedback are.
    Score 0: Completely irrelevant or opposite.
    Score 1: Share some detail, important points differ. (Not COMPLETELY irrelevant)
    Score 2: Share some important point(s), some important points differ.
    Score 3: Share the same important point(s), some details differ.
    Score 4: Really similar.

    These are writing feedback given by two different writing instructor.
    We do not care about the number of the words they used or what tone or format they presented the feedback.
    We care about the how similar they are, conceptually. For instance, if one says "Who is Alex in this passage?" and other says "The story telling is choppy and vague", they are really similar, but not completely.
    
    Instructor 1 feedback: "{feedback1}"

    Instructor 2 feedback: "{feedback2}"

    No explanation at all. Only and only the similarity number. Just the number. Just give an integer between 0 and 4, given the guidelines."""
)

In [ ]:
GPT_SIM_PROMPT = (
    """You are to provide a similarity score in the scale of 0 to 4, as to how similar the two feedback are.
    Score 0: Completely irrelevant or opposite.
    Score 1: Share some detail, important points differ. (Not COMPLETELY irrelevant)
    Score 2: Share some important point(s), but some important points are different/irrelevant.
    Score 3: Share the same important point(s), some details differ.
    Score 4: Really similar.

    These are writing feedback given by two different writing instructor, regarding a paragraph written by a student.
    We do not care about the number of the words they used or what tone or format they presented the feedback.
    We care about the how similar they are, conceptually/semantically.
    
    Samples:
        Instructor A: "This is a great paragraph"
        Instructor B: "Wow! Well-written"
        -> Although the phrasing is very different, they are both saying the same thing. So score=4.
        
        Instructor A: "Who is Alex in this passage?"
        Instructor B: "The story telling is choppy and vague"
        -> They are really similar, but not completely, so score=3

        Instructor A: "This is paragraph is great!"
        Instructor B: "This is well-written, but needs some clarification."
        -> Major point that they share is that they both mention it's a good paragraph, and a major point that they differ is that one mentions it needs clarification. So score=2.
        
        Instructor A: "So the girl didn't notice that she left the apartment?"
        Instructor B: "The second sentence is a little unclear."
        -> These two are not completely irrelevant. The first one is confused about a part of a story, and the second one is saying some sentence is unclear. They are not too similar, but they are not completely irrelevant either. So score=1.
        
        Instructor A: "This is very interesting"
        Instructor B: "The first sentence is a little choppy"
        -> Although they are both feedback towards a paragraph, but they have nothing in common. Completely irrelevant/opposite feedback result in a 0. so score=0.
        
        
    
    
    Instructor A feedback: "{feedback1}"

    Instructor B feedback: "{feedback2}"

    No explanation at all. Only and only the similarity number. Just the number. Just give an integer between 0 and 4, given the guidelines."""
)

## Post-Processing

This section has been omitted from the notebook as it does not affect the main analysis or reported results. A reference implementation of the post-processing procedure is provided separately as a Python file.

You may implement your own post-processing by appending the post-processed version of `['paragraph'][]['model'][0]['feedback'][0]` to the list `['paragraph'][]['model'][0]['feedback']`. Please prefix the appended entry with `"POSTPROCESS:"`.

# UniMatch

## Feedback Segmentation

### GPT API for Segmentation

In [ ]:
# ChatGPT API

from openai import OpenAI
import time
client = OpenAI(api_key=os.environ["OPENAI_KEY"])

In [ ]:
GPT_SEGMENT_TEMPLATE = (
    """An instructor left feedback for a specific paragraph of a student's essay. 
    Your task is to segment this feedback into single feedback points (also called feedback units).

    A single feedback point (also referred to as a feedback unit) is defined as a distinct, self-contained statement that addresses one specific aspect of a student's writing. We treat a feedback point as one or more sentences that convey a coherent thought and can be understood independently . Importantly, a single feedback message may contain multiple such points, each corresponding to a different issue or suggestion.

    Characteristics of a single feedback point:
    - Specificity: Focuses on one aspect of writing (e.g., grammar, word choice, structure, content, tone, a character in the story, etc.).
    - Coherence: Forms a complete thought that is interpretable without requiring additional context.
    - Actionability: It may offer an identifiable issue or suggestion that the student could address.

    If a span elaborates on the same aspect of writing (e.g., further explaining or justifying the same critique or suggestion), it remains part of the same feedback point. If the span shifts to a different aspect, comment, or suggestion, it should be marked as a new feedback point with a delimiter.
    
    Task:
        - Insert $$ (double dollar-sign) as a delimiter when the feedback moves to a new aspect of the student's writing.
        - Each segment should be able to stand alone as a complete feedback point.
        - Do not insert delimiters for mere elaboration, clarification, or examples of the same point.
        - Do not use any other characters (spaces, line breaks, tabs). Only insert $$.
        - If there is a need to place a delimiter in the middle of a sentence, and if the feedback points are
        separated by a transition word (e.g. "and", "but", "also", etc.), then place the delimiter before the
        transition word.
        - Every delimiter used must be either at the end of the feedback, or adjacent to a space (or tab or a
        linebreak character). The only case where a delimiter may be placed between two non-space
        characters is when in the original sample, there is no space after the punctuation of the previous
        feedback point.
    
    Samples:
        Input 1:
            <input>Nice! This is well-written. But you should make it more concise, and be consistent with your characters, like who is Alex here?</input>
        Output 1:
            Nice! This is well-written.$$ But you should make it more concise,$$ and be consistent with your characters, like who is Alex here?$$
        Note 1: There are three feedback units here, one saying this is well-written, another saying make it more concise, and other is talking about characters. So we place a delimiter to segment them.

        Input 2:
            <input>However, the sentence structure is a bit choppy. For example, "As I started to approach, I made eye contact with Tyler." Could use smoother transitions between phrases.</input>
        Output 2:
            However, the sentence structure is a bit choppy. For example, "As I started to approach, I made eye contact with Tyler." Could use smoother transitions between phrases.$$
        Note 2: This is just one feedback unit. Second sentence is just giving an example for the same feedback, and the last sentence is just wrapping up the point. So we just place one delimiter at the end, and nothing in between.
        
        Input 3:
            <input>What did he exactly see? Why did he decide to leave? What about Sara... did she leave too? You need to revise this paragraph.</input>
        Output 3:
            What did he exactly see? Why did he decide to leave? What about Sara... did she leave too? You need to revise this paragraph.$$
        Note 3: This is one feedback point. Although it asks many questions, but basically the feedback is saying that more elaboration and clarification is needed. The writer needs to add details to make it more clear. So we DON'T insert a delimiter for every question. Of course, the questions are asking different things, but this is one feedback unit. They are ALL talking about the same aspect of the essay.
        
        Input 4:
            <input>The sensory details are just great, and the conversation between the characters, really engaging!</input>
        Output 4:
            The sensory details are just great,$$ and the conversation between the characters, really engaging!$$
        Note 4: These are two feedback units, since first is talking about the sensory details and saying it's great, and the second feedback unit is talking about the conversation between the characters being engaging.
        
        Input 5:
            <input>I don't quite understand what did he tell him? What did Sara think about this? Also this paragraph is a little verbose.</input>
        Output 5:
            I don't quite understand what did he tell him? What did Sara think about this?$$ Also this paragraph is a little verbose.$$
        Note 5: We can see that there are only 2 feedback units. First is about something being unclear, and second feedback is saying it's verbose.
        
        Input 6:
            <input>You could add more sensory details. Like explain what was different about the sky that night? These details help your paragraph be more effective.</input>
        Output 6:
            You could add more sensory details. Like explain what was different about the sky that night? These details help your paragraph be more effective.$$
        Note 6: If we place delimiter anywhere except at the end, we would've broken the most important rule, which is the feedback units must be irrelevant. They are related. They are all talking about sensory details. The second sentence is just specifying a specific part, but asking for sensory details (again). The last sentence is just pointing out why sensory details are important. SO WE DON'T PLACE A DELIMITER TO DISTINGUISH THESE SENTENCES AS ALL BELONG TO THE SAME FEEDBACK POINT.
        
        Using the guidelines above, segment the following feedback into feedback units. Don't provide any explanation; just the same input with delimiters placed in correct place(s). 
    Input: 
    <input>{llm_feedback}</input>
    Segmented Feedback:
    """
)

In [ ]:
def llm_segment_feedback(feedback: str, printApiLog: bool=False) -> str:
    """
    Calls GPT to segment a feedback string into feedback points using GPT_SEGMENT_TEMPLATE.
    Retries up to 5 times on failure.
    """
    if printApiLog:
        print("\n\nGPT API SEGMENT:\n", feedback[:30], "...", end=" ")

    for attempt in range(5):
        try:
            response = client.responses.create(
            model="gpt-5-nano",
            reasoning={"effort": "low"},
            input=[
                {
                    "role": "user", 
                    "content": GPT_SEGMENT_TEMPLATE.format(llm_feedback=feedback)
                }
            ]
            )
            out_tmp = response.output_text.strip()

            if printApiLog:
                print("OK")
            return out_tmp

        except Exception as e:
            if printApiLog:
                print(f"\nAttempt {attempt + 1} failed: {e}")
            time.sleep(2)

    if printApiLog:
        print("All GPT attempts failed. Returning original feedback...")
    return feedback  # fallback: return original text unchanged


In [ ]:
import os
import json
from typing import Optional


def segment_files(
    source_dir: str,
    dest_dir: str,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
):
    """
    Copies JSON files from source_dir -> dest_dir.

    For each paragraph where:
      - para["model"] exists and is non-empty
      - para["model"] has exactly ONE item
      - model[0]["feedback"] contains a string starting with "POSTPROCESS: "

    It:
      - extracts the POSTPROCESS content (prefix removed)
      - sends it to llm_segment_feedback(str) -> str
      - appends "FEEDBACK_SEGMENTS" to feedback
      - splits the returned string on "$$"

    All other content is copied as is.
    """

    if not os.path.isdir(source_dir):
        raise FileNotFoundError(f"source_dir not found or not a directory: {source_dir}")

    os.makedirs(dest_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(source_dir) if f.lower().endswith(".json"))

    start = 0 if continue_ind is None else int(continue_ind)
    if start < 0 or start > len(files):
        raise ValueError(f"continue_ind out of range: {start} (num files: {len(files)})")

    if num_files is None:
        selected = files[start:]
    else:
        n = int(num_files)
        if n < 0:
            raise ValueError(f"num_files must be >= 0 or None, got: {n}")
        selected = files[start:n]

    processed_count = 0
    
    sub56_path = "sub56.txt"
    if sub56:
        if not os.path.isfile(sub56_path):
            raise FileNotFoundError(f"{sub56_path} not found")
        with open(sub56_path, "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    for fname in selected:
        print(f"PROCESSING: {processed_count + 1} / {len(selected)} {fname}")
        
        if sub56 and not (fname in sub56_names):
            print("NOT SELECTED... SKIPPED...")
            processed_count +=1
            continue
        print("SELECTED... Continue processing...")

        src_path = os.path.join(source_dir, fname)
        dst_path = os.path.join(dest_dir, fname)

        try:
            with open(src_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"[ERROR] Failed to read {fname}: {e}")
            continue

        paragraphs = data.get("paragraphs", [])
        if not isinstance(paragraphs, list):
            paragraphs = []

        changed = False

        for p_idx, para in enumerate(paragraphs):
            if not isinstance(para, dict):
                continue

            model_list = para.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            model0 = model_list[0]
            if not isinstance(model0, dict):
                continue

            feedback = model0.get("feedback")
            if not isinstance(feedback, list):
                continue

            # Find POSTPROCESS entry
            postprocess_str = None
            for fb in feedback:
                if isinstance(fb, str) and fb.startswith("POSTPROCESS: "):
                    postprocess_str = fb[len("POSTPROCESS: "):]
                    break

            if postprocess_str is None:
                continue

            # Call segmentation function
            try:
                segmented = llm_segment_feedback(postprocess_str, printApiLog)
            except Exception as e:
                print(f"[ERROR] {fname}: llm_segment_feedback failed at paragraph {p_idx}: {e}")
                continue

            if not isinstance(segmented, str):
                continue

            feedback.append("FEEDBACK_SEGMENTS")

            parts = segmented.split("$$")
            for part in parts:
                s = part.strip()
                if len(s) >= 3:
                    feedback.append(s)

            changed = True

        try:
            with open(dst_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"[ERROR] Failed to write {fname}: {e}")
            continue

        processed_count += 1
        print(f"[OK] {fname} -> written ({'changed' if changed else 'no eligible paragraphs'})")

    print(f"\nDone. Wrote {processed_count} file(s) to: {dest_dir}")


In [ ]:
segment_files("source directory",
                   "destination directory",
                   printApiLog = False,
                   num_files=None,
                   continue_ind=None,
                   sub56 = False)

## Feedback Similarity

### API for Similarity

In [ ]:
# ChatGPT API

from openai import OpenAI
import time
import csv

client = OpenAI(api_key= os.environ["OPENAI_KEY"])

In [ ]:
from google import genai
from google.genai import types

GEMINI_API_KEY = os.environ["GEMINI_KEY"]
gem_client = genai.Client(api_key=GEMINI_API_KEY)

#### Main Prompt

In [ ]:
GPT_SIM_PROMPT = (
    """Problem Statement
    Given two feedback units targeted to a paragraph of an essay, how “similar” are they? A feedback unit is defined as a self-contained statement that addresses one specific aspect of a student's writing. 

Guideline
    Given a pair of feedback units, A and B, annotate their “similarity” on a scale of 0 to 4.

    Note 0: We assume that both feedback units are given on the same essay and the same paragraph. If one unit provides a comment without explicitly specifying which part of the story or essay it targets, while the other does, it is safe to assume they refer to the same part, as paragraphs typically cover only a specific portion of the story, essay, or a specific scene.

    Note 1: This is not literal semantic equivalence. Similarity is driven primarily by whether the units target the same aspect/issue and/or they convey the same comment of the writing. Targeting the same aspect establishes a high baseline (~2), and alignment in what the feedback recommends/implies pushes the score up; opposition/contradiction pushes it down toward 0. And vice versa, i.e. they have the same comment, establish a high baseline (~2/3), and alignment in the targeted aspect increases the score towards 4.

    Score 4 is the highest score, indicating that A and B are almost equivalent as feedback. This represents a looser standard than literal semantic equivalence, as the feedback units may be phrased differently, but the underlying “point” they convey is the same.

    Score 0 is the lowest score, indicating that A and B are completely irrelevant to each other or entirely contradictory as feedback.

    Conceptually, scores 1, 2, and 3 are defined as equally spaced points between the two ends of the spectrum: score 0 (completely irrelevant) and score 4 (almost equivalent).

    Score 3 represents that A and B are very close as feedback, but a detail differs. For instance, if they convey the same point (a similar comment), but one of them targets a more specific aspect that is a part of the general aspect the other unit is focusing on. 

    Score 2 indicates that A and B share some important/main points, but also differ on other important/main points. For example, if both focus on the same aspect of the writing but provide different comments, they share an important point (same important aspect) yet differ on the comments (different important points). Similarly, if A and B express the same comment or point but focus on different aspects of the writing, they again share one important point while differing on another.

    Score 1 indicates that A and B are mostly irrelevant as feedback, but share a minor common detail and are not completely unrelated. For instance, both may mention an issue (shared detail), yet the issues themselves are entirely different (differing important points), resulting in a similarity score of 1.
    
    Additional Notes: So we do know that these feedback are given towards the same paragraph of the same essay, we are measuring the similarity of these two feedback units.

Samples of each score:
    Score 4: (Almost) semantically equivalent as feedback units.
    A: <feedback>Great job!</feedback>
    B: <feedback>Wow, this paragraph is really well-written!</feedback>
    -> They both convey generally/all aspects are good. As feedback, neither convey anything else.
    
    A: <feedback>Too many repeated words.</feedback>
    B: <feedback>Try to broaden the vocabulary used.</feedback>
    -> They are just rephrased, and convey the same exact point.
    
    A: <feedback>This opening is captivating!</feedback>
    B: <feedback>Nice job setting the scene and creating suspense.</feedback>
    -> Opening ≈ Setting the Scene (Same aspect). Captivating ≈ Nice job creating suspense (Same point/comment)


    Score 3: Share the same important/main point(s), some details differ.
    A: <feedback>Add more details about the scene.</feedback>
    B: <feedback>What did he see exactly?</feedback>
    -> Both feedback are asking for more details (shared important point), A is asking for general details, B is specifically asking for a specific missing detail (differing detail).
    
    A: <feedback>The story is a little choppy</feedback>
    B: <feedback>I couldn't follow; why didn't she leave then?</feedback>
    -> Both convey the point of the story not having a proper flow and being abrupt (shared important point), B is addressing a specific part of the story that is abrupt.
    
    A: <feedback>This is well-written. Nice job.</feedback>
    B: <feedback>The sensory details are just perfect.</feedback>
    -> They both convey the point of being well-written (shared important point) but one specifies exactly what is making it well-written (different detail).
    
    A: <feedback>This is good…</feedback>
    B: <feedback>Excellent!</feedback>
    -> Both feedback are conveying the point of being well-written (shared important point), but their intensity is different (different detail)


    Score 2: Share some important point(s), some important points differ.
    A: <feedback>Such a balanced and well-written paragraph!</feedback>
    B: <feedback>You well-developed the characters.</feedback>
    -> They both convey the point of being well-written (shared important point), but B specifically mentions a main aspect of the essay, the characters (different important point). 
    Why not 3: Characters of a story cannot be considered as a detail. Characters are part of the paragraph, but the difference is not negligible. Also, the comment of “balanced and well-written” and “well-developed” are also different. 
    
    A: <feedback>Clarify what is exactly happening in this scene!</feedback>
    B: <feedback>Please clarify what you mean by “blue smell”.</feedback>
    -> Both are asking for clarification (shared important point), but B is asking for clarification of a specific phrase (different important point).
    Why not 3: They are asking for clarification, which is very general (but is enough to set the baseline on score 2), but A asks for clarification of what is happening, B is asking for clarification of a phrase or a sensory detail (which is completely different and irrelevant).
    
    A: <feedback>Very nice voice and tone.</feedback>
    B: <feedback>Very well developed characters!</feedback>
    -> They are both saying that an element of the essay is good (important shared point), but both specify a specific aspect of the essay (important different point). That is, the aspect of voice/tone [of the story] is almost completely irrelevant to the establishment of the characters.


    Score 1: Share a few details, but important points differ. (Not COMPLETELY irrelevant, or they share the same general feedback category)
    General category: Mistake, Elaboration/Clarification, Praise
    A: <feedback>It should be “wear”, not “where”</feedback>
    B: <feedback>Too much repetition...</feedback>
    -> They share the same feedback category -- “mistake”
    
    A: <feedback>Interesting conversation between you and Sara!</feedback>
    B: <feedback>Nice job with the sensory details.</feedback>
    -> Same feedback category -- “praise”, but the focus and the main elements of the feedback is irrelevant: sensory details of the scene, and the conversation between characters. Feedback A also only conveys the point that the conversation itself is interesting, and is not specifically saying this is well-written.


    Score 0: Completely irrelevant or even opposite. (They neither have the same content nor share the same aspect, or they contradict each other.)
    A: <feedback>This paragraph is great!</feedback>
    B: <feedback>This paragraph is a little choppy</feedback>
    -> They somewhat contradict each other. They also don't share the same feedback category.
    
    A: <feedback>Very well-written.</feedback>
    B: <feedback>So why did she remain on the couch?</feedback>
    -> Irrelevant: A is praising, B is asking for clarification about some part of the story.
    
    A: <feedback>So much authenticity in your voice!</feedback>
    B: <feedback>You can reduce the exaggeration here</feedback>
    -> Irrelevant: In fact, A is praising the paragraph, while B is criticizing an aspect of the paragraph.


    
    Instructor A feedback: <feedback>{feedback1}</feedback>

    Instructor B feedback: <feedback>{feedback2}</feedback>

    No explanation at all. Only and only the similarity number. Just give an integer between 0 and 4, given the guidelines."""
)

#### Buffer Prompt

In [ ]:
# BATCHING SYSTEM

GPT_SIM_BATCH_PROMPT = """Problem Statement
Given two feedback units targeted to a paragraph of an essay, how “similar” are they? A feedback unit is defined as a self-contained statement that addresses one specific aspect of a student's writing. 

Guideline
Given a pair of feedback units, A and B, annotate their “similarity” on a scale of 0 to 4.

Note 0: We assume that both feedback units are given on the same essay and the same paragraph. If one unit provides a comment without explicitly specifying which part of the story or essay it targets, while the other does, it is safe to assume they refer to the same part, as paragraphs typically cover only a specific portion of the story, essay, or a specific scene.

Note 1: This is not literal semantic equivalence. Similarity is driven primarily by whether the units target the same aspect/issue and/or they convey the same comment of the writing. Targeting the same aspect establishes a high baseline (~2), and alignment in what the feedback recommends/implies pushes the score up; opposition/contradiction pushes it down toward 0. And vice versa, i.e. they have the same comment, establish a high baseline (~2/3), and alignment in the targeted aspect increases the score towards 4.

Score 4 is the highest score, indicating that A and B are almost equivalent as feedback. This represents a looser standard than literal semantic equivalence, as the feedback units may be phrased differently, but the underlying “point” they convey is the same.

Score 0 is the lowest score, indicating that A and B are completely irrelevant to each other or entirely contradictory as feedback.

Conceptually, scores 1, 2, and 3 are defined as equally spaced points between the two ends of the spectrum: score 0 (completely irrelevant) and score 4 (almost equivalent).

Score 3 represents that A and B are very close as feedback, but a detail differs. For instance, if they convey the same point (a similar comment), but one of them targets a more specific aspect that is a part of the general aspect the other unit is focusing on. 

Score 2 indicates that A and B share some important/main points, but also differ on other important/main points. For example, if both focus on the same aspect of the writing but provide different comments, they share an important point (same important aspect) yet differ on the comments (different important points). Similarly, if A and B express the same comment or point but focus on different aspects of the writing, they again share one important point while differing on another.

Score 1 indicates that A and B are mostly irrelevant as feedback, but share a minor common detail and are not completely unrelated. For instance, both may mention an issue (shared detail), yet the issues themselves are entirely different (differing important points), resulting in a similarity score of 1.

Additional Notes: So we do know that these feedback are given towards the same paragraph of the same essay, we are measuring the similarity of these two feedback units.

Samples of each score:
Score 4: (Almost) semantically equivalent as feedback units.
A: <feedback>Great job!</feedback>
B: <feedback>Wow, this paragraph is really well-written!</feedback>
-> They both convey generally/all aspects are good. As feedback, neither convey anything else.

A: <feedback>Too many repeated words.</feedback>
B: <feedback>Try to broaden the vocabulary used.</feedback>
-> They are just rephrased, and convey the same exact point.

A: <feedback>This opening is captivating!</feedback>
B: <feedback>Nice job setting the scene and creating suspense.</feedback>
-> Opening ≈ Setting the Scene (Same aspect). Captivating ≈ Nice job creating suspense (Same point/comment)

Score 3: Share the same important/main point(s), some details differ.
A: <feedback>Add more details about the scene.</feedback>
B: <feedback>What did he see exactly?</feedback>
-> Both feedback are asking for more details (shared important point), A is asking for general details, B is specifically asking for a specific missing detail (differing detail).

A: <feedback>The story is a little choppy</feedback>
B: <feedback>I couldn't follow; why didn't she leave then?</feedback>
-> Both convey the point of the story not having a proper flow and being abrupt (shared important point), B is addressing a specific part of the story that is abrupt.

A: <feedback>This is well-written. Nice job.</feedback>
B: <feedback>The sensory details are just perfect.</feedback>
-> They both convey the point of being well-written (shared important point) but one specifies exactly what is making it well-written (different detail).

A: <feedback>This is good…</feedback>
B: <feedback>Excellent!</feedback>
-> Both feedback are conveying the point of being well-written (shared important point), but their intensity is different (different detail)

Score 2: Share some important point(s), some important points differ.
A: <feedback>Such a balanced and well-written paragraph!</feedback>
B: <feedback>You well-developed the characters.</feedback>
-> They both convey the point of being well-written (shared important point), but B specifically mentions a main aspect of the essay, the characters (different important point).
Why not 3: Characters of a story cannot be considered as a detail. Characters are part of the paragraph, but the difference is not negligible. Also, the comment of “balanced and well-written” and “well-developed” are also different.

A: <feedback>Clarify what is exactly happening in this scene!</feedback>
B: <feedback>Please clarify what you mean by “blue smell”.</feedback>
-> Both are asking for clarification (shared important point), but B is asking for clarification of a specific phrase (different important point).
Why not 3: They are asking for clarification, which is very general (but is enough to set the baseline on score 2), but A asks for clarification of what is happening, B is asking for clarification of a phrase or a sensory detail (which is completely different and irrelevant).

A: <feedback>Very nice voice and tone.</feedback>
B: <feedback>Very well developed characters!</feedback>
-> They are both saying that an element of the essay is good (important shared point), but both specify a specific aspect of the essay (important different point). That is, the aspect of voice/tone [of the story] is almost completely irrelevant to the establishment of the characters.

Score 1: Share a few details, but important points differ. (Not COMPLETELY irrelevant, or they share the same general feedback category)
General category: Mistake, Elaboration/Clarification, Praise
A: <feedback>It should be “wear”, not “where”</feedback>
B: <feedback>Too much repetition...</feedback>
-> They share the same feedback category -- “mistake”

A: <feedback>Interesting conversation between you and Sara!</feedback>
B: <feedback>Nice job with the sensory details.</feedback>
-> Same feedback category -- “praise”, but the focus and the main elements of the feedback is irrelevant: sensory details of the scene, and the conversation between characters. Feedback A also only conveys the point that the conversation itself is interesting, and is not specifically saying this is well-written.

Score 0: Completely irrelevant or even opposite.
A: <feedback>This paragraph is great!</feedback>
B: <feedback>This paragraph is a little choppy</feedback>
-> They somewhat contradict each other. They also don't share the same feedback category.

A: <feedback>Very well-written.</feedback>
B: <feedback>So why did she remain on the couch?</feedback>
-> Irrelevant: A is praising, B is asking for clarification about some part of the story.

A: <feedback>So much authenticity in your voice!</feedback>
B: <feedback>You can reduce the exaggeration here</feedback>
-> Irrelevant: A is praising the paragraph, while B is criticizing an aspect of the paragraph.

Now score EACH pair completely INDEPENDENTLY. Do NOT let the score of one pair influence another.

You will receive multiple labeled pairs. For EACH label, output exactly one line:

<label>) <integer 0-4>

No explanation. No extra text. Only those lines, one per label, in the same order.

PAIRS:
{pairs_block}
"""

GEMINI_SIM_BATCH_PROMPT=GPT_SIM_BATCH_PROMPT

### Gemini API Function

In [ ]:
import re
import time
from typing import Union

from google import genai
from google.genai import types


gem_client = genai.Client(api_key=GEMINI_API_KEY)


GEMINI_SIM_PROMPT = GPT_SIM_PROMPT


def _extract_score_0_to_4(text: str) -> int:
    """
    Extract a single integer 0-4 from Gemini output.
    Returns 0 if invalid.
    """
    if not text:
        return 0

    s = text.strip()

    # Best case: model obeys instruction
    if re.fullmatch(r"[0-4]", s):
        return int(s)

    # Fallback: find a standalone digit 0-4
    m = re.search(r"(?<!\d)([0-4])(?!\d)", s)
    return int(m.group(1)) if m else 0


def gemini_similarity_score(
    feedback1: str,
    feedback2: str,
    printApiLog: bool = False,
    scoreOnly: bool = True,
) -> Union[int, str]:
    """
    Calls Gemini API to compute similarity score between two feedback strings.
    Returns an integer 0-4.
    Returns 0 if the API fails or response is invalid.
    """
    if printApiLog:
        print("GEMINI SIM:", feedback1[:20], "---", feedback2[:20], end=" ")

    prompt = GEMINI_SIM_PROMPT.format(
        feedback1=str(feedback1),
        feedback2=str(feedback2),
    )

    for attempt in range(5):
        try:
            response = gem_client.models.generate_content(
                model="gemini-3-flash-preview",
                #model = "gemini-2.5-flash",
                contents=prompt,
                #config=types.GenerateContentConfig(
                #    thinking_config=types.ThinkingConfig(
                #        thinking_level="low"
                #    )
                #),
            )

            out_text = (response.text or "").strip()

            if not scoreOnly:
                return out_text

            score = _extract_score_0_to_4(out_text)

            if printApiLog:
                print(score if score in range(5) else "Bad:", out_text)

            return score if score in range(5) else 0

        except Exception as e:
            if printApiLog:
                print(f"Attempt {attempt + 1} failed:", e)
            time.sleep(2)

    if printApiLog:
        print("All Gemini attempts failed. Returning 0.")

    return 0


### GPT API Function

In [ ]:
# Final Model to check
def gpt_similarity_score(
    feedback1: str,
    feedback2: str,
    printApiLog: bool = False,
    scoreOnly: bool = True
) -> int:
    
    GPT_MODEL = "gpt-5.1"
    print("GPT Model:", GPT_MODEL, end=' ')

    if printApiLog:
        print("GPT API SIM:", feedback1[:20], "---", feedback2[:20], end=" ")

    for attempt in range(5):
        try:
            response = client.responses.create(
                model=GPT_MODEL,
                #model="gpt-5.1",
                reasoning={"effort": "medium"},
                #temperature=0.1,
                #max_output_tokens=10,
                #top_p=1,
                input=[
                    {
                        "role": "user",
                        "content": GPT_SIM_PROMPT.format(
                            feedback1=feedback1,
                            feedback2=feedback2
                        ),
                    }
                ],
            )

            # STRICT extraction
            out_text = response.output_text.strip()

            if not scoreOnly:
                return out_text

            if out_text in {"0", "1", "2", "3", "4"}:
                score = int(out_text)
                if printApiLog:
                    print(score)
                return score

            if printApiLog:
                print("Bad response:", out_text)

            return 0

        except Exception as e:
            if printApiLog:
                print(f"Attempt {attempt + 1} failed:", e)
            time.sleep(2)

    if printApiLog:
        print("All attempts failed. Returning 0.")
    return 0


### GPT Buffer Function

In [ ]:
import re
import time
from collections import deque
from typing import Deque, List, Optional, Tuple


def _make_tags(n: int) -> List[str]:
    # A, B, ..., Z, AA, AB, ...
    tags: List[str] = []
    i = 0
    while len(tags) < n:
        x = i
        s = ""
        while True:
            s = chr(ord("A") + (x % 26)) + s
            x = x // 26 - 1
            if x < 0:
                break
        tags.append(s)
        i += 1
    return tags


def _build_pairs_block(pairs: List[Tuple[str, str]]) -> str:
    tags = _make_tags(len(pairs))
    chunks: List[str] = []
    for tag, (f1, f2) in zip(tags, pairs):
        chunks.append(
            f"{tag})\n"
            f"Instructor A feedback: <feedback>{f1}</feedback>\n"
            f"Instructor B feedback: <feedback>{f2}</feedback>\n"
        )
    return "\n".join(chunks)


def _parse_tagged_scores(text: str, expected_tags: List[str]) -> Optional[List[int]]:
    # STRICT: reject duplicates, missing tags, extra tags.
    scores_by_tag: dict[str, int] = {}

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        m = re.match(r"^([A-Z]{1,10})\)\s*([0-4])\s*$", line)
        if not m:
            continue

        tag, score_str = m.group(1), m.group(2)

        # reject duplicates
        if tag in scores_by_tag:
            return None

        scores_by_tag[tag] = int(score_str)

    if len(scores_by_tag) != len(expected_tags):
        return None

    if set(scores_by_tag.keys()) != set(expected_tags):
        return None

    return [scores_by_tag[tag] for tag in expected_tags]


class GPTSimilarityBuffered:
    """
    Buffers (feedback1, feedback2) pairs; sends to GPT when buffer hits buffer_size
    or when flush=True is passed.

    add_pair(...) returns a LIST of newly-produced scores (possibly empty).
    Scores are returned in the same order as inputs were provided.
    """

    def __init__(
        self,
        client,
        buffer_size: int,
        model: str = "gpt-5-nano",
        printApiLog: bool = False,
        max_attempts: int = 5,
        max_parse_retries: int = 3,
    ):
        if buffer_size <= 0:
            raise ValueError("buffer_size must be >= 1")
        if max_attempts <= 0:
            raise ValueError("max_attempts must be >= 1")
        if max_parse_retries <= 0:
            raise ValueError("max_parse_retries must be >= 1")

        self.client = client
        self.buffer_size = buffer_size
        self.model = model
        self.printApiLog = printApiLog
        self.max_attempts = max_attempts
        self.max_parse_retries = max_parse_retries

        self._buffer: List[Tuple[str, str]] = []
        self._pending: Deque[int] = deque()

    def add_pair(
        self,
        feedback1: Optional[str],
        feedback2: Optional[str],
        flush: bool = False
    ) -> List[int]:
        produced: List[int] = []

        # Drain any pending scores first
        while self._pending:
            produced.append(self._pending.popleft())

        # Add current pair unless this call is flush-only
        if feedback1 is not None and feedback2 is not None:
            self._buffer.append((feedback1, feedback2))

        should_send = flush or (len(self._buffer) >= self.buffer_size)
        if not should_send:
            if self.printApiLog:
                print(feedback1[:10], '----', feedback2[:10], " Buffered...")
            return produced  # likely empty

        # If flush=True and buffer is empty, nothing to do
        if not self._buffer:
            return produced

        batch_pairs = self._buffer
        self._buffer = []

        tags = _make_tags(len(batch_pairs))
        pairs_block = _build_pairs_block(batch_pairs)
        prompt_text = GPT_SIM_BATCH_PROMPT.format(pairs_block=pairs_block)

        if self.printApiLog:
            print(f"GPT Model: {self.model} | batch={len(batch_pairs)}")

        parsed: Optional[List[int]] = None
        parse_tries = 0

        for attempt in range(self.max_attempts):
            try:
                response = self.client.responses.create(
                    model=self.model,
                    #model="gpt-5.1",
                    reasoning={"effort": "low"},
                    #temperature=0.1,
                    #max_output_tokens=10,
                    #top_p=1,
                    input=[{"role": "user", "content": prompt_text}],
                )
                out_text = response.output_text.strip()

                parsed = _parse_tagged_scores(out_text, tags)
                if parsed is None:
                    parse_tries += 1
                    if self.printApiLog:
                        preview = out_text.replace("\n", " ")[:200]
                        print(
                            f"Bad parse (parse_try {parse_tries}/{self.max_parse_retries}) | {preview}"
                        )

                    # Retry parse by re-calling the model
                    if parse_tries < self.max_parse_retries:
                        time.sleep(1)
                        continue

                    # Too many bad parses -> fallback zeros
                    parsed = [0] * len(batch_pairs)

                # Valid parsed (or fallback zeros)
                for s in parsed:
                    self._pending.append(s)
                break

            except Exception as e:
                if self.printApiLog:
                    print(f"Attempt {attempt + 1} failed:", e)
                time.sleep(2)

        if parsed is None:
            # All API attempts failed
            for _ in batch_pairs:
                self._pending.append(0)

        while self._pending:
            produced.append(self._pending.popleft())

        return produced


### Gemini Buffer Function

In [ ]:
import re
import time
from collections import deque
from typing import Deque, List, Optional, Tuple


def _make_tags(n: int) -> List[str]:
    # A, B, ..., Z, AA, AB, ...
    tags: List[str] = []
    i = 0
    while len(tags) < n:
        x = i
        s = ""
        while True:
            s = chr(ord("A") + (x % 26)) + s
            x = x // 26 - 1
            if x < 0:
                break
        tags.append(s)
        i += 1
    return tags


def _build_pairs_block(pairs: List[Tuple[str, str]]) -> str:
    tags = _make_tags(len(pairs))
    chunks: List[str] = []
    for tag, (f1, f2) in zip(tags, pairs):
        chunks.append(
            f"{tag})\n"
            f"Instructor A feedback: <feedback>{f1}</feedback>\n"
            f"Instructor B feedback: <feedback>{f2}</feedback>\n"
        )
    return "\n".join(chunks)


def _parse_tagged_scores(text: str, expected_tags: List[str]) -> Optional[List[int]]:
    # STRICT: reject duplicates, missing tags, extra tags.
    scores_by_tag: dict[str, int] = {}

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        m = re.match(r"^([A-Z]{1,10})\)\s*([0-4])\s*$", line)
        if not m:
            continue

        tag, score_str = m.group(1), m.group(2)

        # reject duplicates
        if tag in scores_by_tag:
            return None

        scores_by_tag[tag] = int(score_str)

    if len(scores_by_tag) != len(expected_tags):
        return None

    if set(scores_by_tag.keys()) != set(expected_tags):
        return None

    return [scores_by_tag[tag] for tag in expected_tags]


class GeminiSimilarityBuffered:
    """
    Buffers (feedback1, feedback2) pairs; sends to Gemini when buffer hits buffer_size
    or when flush=True is passed.

    add_pair(...) returns a LIST of newly-produced scores (possibly empty).
    Scores are returned in the same order as inputs were provided.
    """

    def __init__(
        self,
        gem_client,
        buffer_size: int,
        #model: str = "gemini-3-flash-preview",
        model: str = 'gemini-3.1-flash-lite-preview',
        prompt_template: str = GEMINI_SIM_BATCH_PROMPT, 
        printApiLog: bool = False,
        max_attempts: int = 5,
        max_parse_retries: int = 3,
        sleep_between_attempts_s: float = 2.0,
        sleep_between_parse_retries_s: float = 1.0,
    ):
        if buffer_size <= 0:
            raise ValueError("buffer_size must be >= 1")
        if max_attempts <= 0:
            raise ValueError("max_attempts must be >= 1")
        if max_parse_retries <= 0:
            raise ValueError("max_parse_retries must be >= 1")
        if not prompt_template:
            raise ValueError("prompt_template must be a non-empty string")

        self.gem_client = gem_client
        self.buffer_size = buffer_size
        self.model = model
        self.prompt_template = prompt_template
        self.printApiLog = printApiLog
        self.max_attempts = max_attempts
        self.max_parse_retries = max_parse_retries
        self.sleep_between_attempts_s = sleep_between_attempts_s
        self.sleep_between_parse_retries_s = sleep_between_parse_retries_s

        self._buffer: List[Tuple[str, str]] = []
        self._pending: Deque[int] = deque()

    def _extract_text(self, response) -> str:
        t = getattr(response, "text", None)
        if isinstance(t, str):
            return t.strip()

        # Some SDK variants return a list of candidates/parts
        try:
            return str(response).strip()
        except Exception:
            return ""

    def add_pair(
        self,
        feedback1: Optional[str],
        feedback2: Optional[str],
        flush: bool = False,
    ) -> List[int]:
        produced: List[int] = []

        # Drain any pending scores first
        while self._pending:
            produced.append(self._pending.popleft())

        # Add current pair unless this call is flush-only
        if feedback1 is not None and feedback2 is not None:
            self._buffer.append((feedback1, feedback2))

        should_send = flush or (len(self._buffer) >= self.buffer_size)
        if not should_send:
            if self.printApiLog and feedback1 is not None and feedback2 is not None:
                print(feedback1[:10], "----", feedback2[:10], " Buffered...")
            return produced  # empty... probably

        # If flush=True and buffer is empty, nothing to do
        if not self._buffer:
            return produced

        batch_pairs = self._buffer
        self._buffer = []

        tags = _make_tags(len(batch_pairs))
        pairs_block = _build_pairs_block(batch_pairs)
        prompt_text = self.prompt_template.format(pairs_block=pairs_block)
        #print("PROMPT:", prompt_text)

        if self.printApiLog:
            print(f"GEMINI Model: {self.model} | batch={len(batch_pairs)}")

        parsed: Optional[List[int]] = None
        parse_tries = 0

        for attempt in range(self.max_attempts):
            try:
                response = self.gem_client.models.generate_content(
                    model=self.model,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        thinking_config=types.ThinkingConfig(
                            thinking_level="minimal"
                        )
                    ),
                )
                #print("PROMPT:", prompt_text)
                out_text = self._extract_text(response)
                #print("OUTPUT:", out_text)

                parsed = _parse_tagged_scores(out_text, tags)
                if parsed is None:
                    parse_tries += 1
                    if self.printApiLog:
                        preview = out_text.replace("\n", " ")[:200]
                        print(
                            f"Bad parse (parse_try {parse_tries}/{self.max_parse_retries}) | {preview}"
                        )

                    # Retry parse by re-calling the model
                    if parse_tries < self.max_parse_retries:
                        time.sleep(self.sleep_between_parse_retries_s)
                        continue

                    # Too many bad parses -> fallback zeros
                    parsed = [0] * len(batch_pairs)

                # Valid parsed (or fallback zeros)
                for s in parsed:
                    self._pending.append(s)
                break

            except Exception as e:
                if self.printApiLog:
                    print(f"Attempt {attempt + 1} failed:", e)
                time.sleep(self.sleep_between_attempts_s)

        if parsed is None:
            # All API attempts failed
            for _ in batch_pairs:
                self._pending.append(0)

        while self._pending:
            produced.append(self._pending.popleft())

        return produced



## UniMatch (Similarity and Matching)

### Threshold-Based

Buffering is not implemented for the threshold-based UniMatch, as this approach was primarily used as an auxiliary method.

However, in the threshold-free subsection of this notebook, buffering is implemented. Instead of using Hungarian matching (which is applied there to compute soft p/r/f1 scores), you can easily adapt the same setup to implement threshold-based UniMatch.

Note that buffering is applied at the API call level, and soft metrics are computed after all API calls are completed and the results are saved to a file for a given configuration.

This design allows you to reuse the stored results to evaluate performance across different thresholds, and experiment with alternative metrics without rerunning the API calls.

In [ ]:
import os
import json
from typing import Optional, Dict, List


def unimatch(
    model_source: str,
    annot_source: str,
    dest_dir: str,
    match_threshold: int = 2,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
):
    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")

    os.makedirs(dest_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))

    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    # ---- Global accumulators ----
    micro_TP = micro_FP = micro_FN = 0
    macro_precisions = []
    macro_recalls = []
    macro_f1s = []

    for i, fname in enumerate(selected, 1):
        print(f"PROCESSING: {i}/{len(selected)} {fname}")

        if sub56 and fname not in sub56_names:
            print("SKIPPED...")
            continue

        model_path = os.path.join(model_source, fname)
        annot_path = os.path.join(annot_source, fname)
        dst_path = os.path.join(dest_dir, fname)

        if not os.path.isfile(annot_path):
            print(f"[ERROR] Missing annotation file: {fname}")
            continue

        with open(model_path, "r", encoding="utf-8") as f:
            model_data = json.load(f)
        with open(annot_path, "r", encoding="utf-8") as f:
            annot_data = json.load(f)

        for idx, (p_model, p_annot) in enumerate(
            zip(model_data.get("paragraphs", []),
                annot_data.get("paragraphs", []))
        ):
            model_list = p_model.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            feedback = model_list[0].get("feedback")
            if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                continue

            seg_idx = feedback.index("FEEDBACK_SEGMENTS")
            model_segs = [
                s for s in feedback[seg_idx + 1:]
                if isinstance(s, str) and len(s.strip()) >= 3
            ]
            if not model_segs:
                continue

            comments = [
                a.get("comment", "").strip()
                for a in p_annot.get("annotations", [])
                if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
            ]
            if not comments:
                continue
            
            if one_GT_only and (not isinstance(comments, list) or len(comments) != 1):
                continue

            # ---- Matching ----
            model_matched = [False] * len(model_segs)
            annot_matched = [False] * len(comments)
            segsim: List[Dict] = []

            for mi, ms in enumerate(model_segs):
                for ai, ac in enumerate(comments):
                    score = gpt_similarity_score(ms, ac, printApiLog, scoreOnly=True)
                    segsim.append({
                        "feedback": ms,
                        "comment": ac,
                        "score": score,
                    })
                    if score >= match_threshold:
                        model_matched[mi] = True
                        annot_matched[ai] = True

            TP = sum(model_matched)
            FP = len(model_segs) - TP
            FN = annot_matched.count(False)

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

            print({
                "file": fname,
                "paragraph": idx,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            })

            micro_TP += TP
            micro_FP += FP
            micro_FN += FN

            macro_precisions.append(precision)
            macro_recalls.append(recall)
            macro_f1s.append(f1)

            p_model["segsim"] = segsim

        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(model_data, f, ensure_ascii=False, indent=2)

    # ---- Final metrics ----
    micro_precision = micro_TP / (micro_TP + micro_FP) if (micro_TP + micro_FP) > 0 else 0.0
    micro_recall = micro_TP / (micro_TP + micro_FN) if (micro_TP + micro_FN) > 0 else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0 else 0.0
    )

    print("\n FINAL METRICS ")
    print({
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": sum(macro_precisions) / len(macro_precisions) if macro_precisions else 0.0,
        "macro_recall": sum(macro_recalls) / len(macro_recalls) if macro_recalls else 0.0,
        "macro_f1": sum(macro_f1s) / len(macro_f1s) if macro_f1s else 0.0,
    })


In [ ]:
unimatch(
    model_source= './segment_individual/seg_s30/',
    annot_source= './segment_individual/seg_annot/',
    dest_dir= './unimatch_individual/eval_s30',
    match_threshold = 2,
    num_files = 50,
    continue_ind = None,
    printApiLog = False,
    sub56 = True,
    one_GT_only = True,
)

### Gemini Similarity (Batch API)

This cell is provided as a convenience if you choose to use batched API calls. Note that this is not the same as the buffering approach introduced in the paper.

Batching operates solely at the API call level (a.k.a offline API execution) and is included here for reference purposes only. It was not used in the main experiments, as combining batching with our buffering approach can become complex, specially when the model produces malformed outputs.

#### Stage 1 (Creating and submitting a batch)

In [ ]:
# PROMPT & GEMINI KEY MUST BE AVAILABLE

import os
import json
import time
import re
import hashlib
from typing import Optional, Dict, List, Tuple, Union

from google import genai
from google.genai import types



gem_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_SIM_PROMPT = GPT_SIM_PROMPT



def _stable_pair_key(feedback1: str, feedback2: str) -> str:
    """
    Deterministic key so Stage 2 can find the exact output using ONLY (feedback1, feedback2)

    Uses SHA-256 over a delimiter-separated pair. Collision risk is negligible in practice.
    """
    f1 = "" if feedback1 is None else str(feedback1)
    f2 = "" if feedback2 is None else str(feedback2)
    h = hashlib.sha256((f1 + "\u0000" + f2).encode("utf-8")).hexdigest()
    return h  # keep full hash for safety


def _write_jsonl_line(fp, obj: Dict) -> None:
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")


def _upload_jsonl_with_fallback(
    client: genai.Client,
    path: str,
    display_name: str,
) -> types.File:
    """
    Official docs show mime_type='jsonl'. If that fails (SDK/version-dependent),
    try a couple of common alternatives.
    """
    mime_candidates = ["jsonl", "application/jsonl", "application/octet-stream", "text/plain"]
    last_err = None

    for mt in mime_candidates:
        try:
            return client.files.upload(
                file=path,
                config=types.UploadFileConfig(display_name=display_name, mime_type=mt),
            )
        except Exception as e:
            last_err = e

    # If all fail, raise the last error
    raise last_err


def build_and_submit_unimatch_batch(
    *,
    client: genai.Client,
    model: str,
    model_source: str,
    annot_source: str,
    out_dir: str,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    sub56: bool = False,
    one_GT_only: bool = False,
    display_name: str = "unimatch-batch",
) -> Dict[str, str]:
    """
    Stage 1:
      - Scans your jsons with (nearly) the same logic as unimatch
      - Creates:
          out_dir/batch_requests.jsonl  (upload this)
          out_dir/batch_manifest.jsonl  (sanity + mapping)
      - Uploads batch_requests.jsonl and creates batch job
      - Returns { 'job_name': ..., 'uploaded_file': ..., 'requests_path': ..., 'manifest_path': ... }
    """
    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")
    os.makedirs(out_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    if start < 0 or start > len(files):
        raise ValueError("continue_ind out of range")

    selected = files[start:] if num_files is None else files[start:start + int(num_files)]
    #print(selected)
    #return

    sub56_names = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    requests_path = os.path.join(out_dir, "batch_requests.jsonl")
    manifest_path = os.path.join(out_dir, "batch_manifest.jsonl")

    n_written = 0
    with open(requests_path, "w", encoding="utf-8") as req_fp, open(manifest_path, "w", encoding="utf-8") as man_fp:
        for fname in selected:
            if sub56 and sub56_names is not None and fname not in sub56_names:
                continue

            model_path = os.path.join(model_source, fname)
            annot_path = os.path.join(annot_source, fname)
            if not os.path.isfile(annot_path):
                continue

            with open(model_path, "r", encoding="utf-8") as f:
                model_data = json.load(f)
            with open(annot_path, "r", encoding="utf-8") as f:
                annot_data = json.load(f)

            paragraphs_model = model_data.get("paragraphs", [])
            paragraphs_annot = annot_data.get("paragraphs", [])

            for p_idx, (p_model, p_annot) in enumerate(zip(paragraphs_model, paragraphs_annot)):
                annots = p_annot.get("annotations", [])
                if one_GT_only and (not isinstance(annots, list) or len(annots) != 1):
                    continue

                model_list = p_model.get("model")
                if not isinstance(model_list, list) or len(model_list) != 1:
                    continue

                feedback = model_list[0].get("feedback")
                if not isinstance(feedback, list):
                    continue
                if "FEEDBACK_SEGMENTS" not in feedback:
                    continue

                seg_idx = feedback.index("FEEDBACK_SEGMENTS")
                model_segs = [s for s in feedback[seg_idx + 1:] if isinstance(s, str) and len(s.strip()) >= 3]
                if not model_segs:
                    continue

                comments = [
                    a.get("comment", "").strip()
                    for a in annots
                    if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
                ]
                if not comments:
                    continue

                for ms in model_segs:
                    for ac in comments:
                        key = _stable_pair_key(ms, ac)

                        prompt = GEMINI_SIM_PROMPT.format(feedback1=str(ms), feedback2=str(ac))

                        # Batch JSONL line format from docs:
                        # {"key": "...", "request": { ...GenerateContentRequest... }}
                        _write_jsonl_line(req_fp, {
                            "key": key,
                            "request": {
                                "contents": [{
                                    "role": "user",
                                    "parts": [{"text": prompt}],
                                }],
                            }
                        })

                        # Separate manifest line for sanity checks + metadata
                        _write_jsonl_line(man_fp, {
                            "key": key,
                            "file": fname,
                            "paragraph": p_idx,
                            "feedback_segment": ms,
                            "annotation_comment": ac,
                        })

                        n_written += 1

    if n_written == 0:
        raise RuntimeError("No requests were written. Check your filters / paths / one_GT_only / sub56.")
    #print("Number of requests: ", n_written)

    uploaded = _upload_jsonl_with_fallback(client, requests_path, display_name=display_name)

    batch_job = client.batches.create(
        model=model,
        src=uploaded.name,  # file-based
        config={"display_name": display_name},
    )

    return {
        "job_name": batch_job.name,
        "uploaded_file": uploaded.name,
        "requests_path": requests_path,
        "manifest_path": manifest_path,
        "n_written": n_written,
    }


In [ ]:
setting = 's3' 


info = build_and_submit_unimatch_batch(
    client=gem_client,
    model="gemini-3-flash-preview",
    model_source='./segment_individual/seg_'+setting,
    annot_source='./segment_individual/seg_annot/',
    out_dir='./unimatch_batches/eval_'+setting,
    num_files=None,
    continue_ind=None,
    sub56=False,
    one_GT_only=True,
    display_name=f"unimatch-{setting}",
)
print(info)

In [ ]:
# Run on many

In [ ]:
settings_to_batch = [3, 5, 37, 39, 38, 7, 8, 10, 12, 43, 45, 44, 14, 15, 16, 17, 18, 19, 20, 21, 51, 40, 42, 41, 22, 23, 24, 46, 48, 47, 25, 27, 28, 30, 35, 36]

In [ ]:
batches = []

for x in settings_to_batch:
    setting = 's'+str(x)
    info = build_and_submit_unimatch_batch(
        client=gem_client,
        model="gemini-3-flash-preview",
        model_source='./segment_individual/seg_'+setting,
        annot_source='./segment_individual/seg_annot/',
        out_dir='./unimatch_batches/eval_'+setting,
        num_files=None,
        continue_ind=None,
        sub56=False,
        one_GT_only=True,
        display_name=f"unimatch-{setting}",
    )
    print(info)
    batches.append(info)

In [ ]:
job = gem_client.batches.get(name="batches/JOBID")
print(job.state.name)

In [ ]:
import os
import json
import time
import re
import hashlib
from typing import Dict, Optional, Union, Callable, Any

from google import genai


def _extract_score_0_to_4(text: str) -> int:
    if not text:
        return 0
    s = text.strip()
    if re.fullmatch(r"[0-4]", s):
        return int(s)
    m = re.search(r"(?<!\d)([0-4])(?!\d)", s)
    return int(m.group(1)) if m else 0


def _stable_pair_key(feedback1: str, feedback2: str) -> str:
    f1 = "" if feedback1 is None else str(feedback1)
    f2 = "" if feedback2 is None else str(feedback2)
    return hashlib.sha256((f1 + "\u0000" + f2).encode("utf-8")).hexdigest()


def _first_text_from_generate_content_response(resp: Dict[str, Any]) -> str:
    """
    Extracts text from a GenerateContentResponse JSON object.
    Standard path is candidates[0].content.parts[*].text.
    """
    try:
        cands = resp.get("candidates") or []
        if not cands:
            return ""
        content = cands[0].get("content") or {}
        parts = content.get("parts") or []
        for p in parts:
            t = p.get("text")
            if isinstance(t, str) and t.strip():
                return t
        return ""
    except Exception:
        return ""


def poll_and_download_batch_results(
    *,
    client: genai.Client,
    job_name: str,
    out_jsonl_path: str,
    poll_seconds: int = 10,
) -> str:
    """
    Polls job until it reaches a terminal state.
    If succeeded and file output exists, downloads.
    """
    completed = {
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED",
    }

    batch_job = client.batches.get(name=job_name)
    while batch_job.state.name not in completed:
        time.sleep(poll_seconds)
        batch_job = client.batches.get(name=job_name)

    if batch_job.state.name != "JOB_STATE_SUCCEEDED":
        raise RuntimeError(f"Batch job did not succeed: state={batch_job.state.name}, error={batch_job.error}")

    if not (batch_job.dest and batch_job.dest.file_name):
        raise RuntimeError("Succeeded job has no dest.file_name (no file results found).")

    result_file_name = batch_job.dest.file_name
    blob = client.files.download(file=result_file_name) 

    os.makedirs(os.path.dirname(out_jsonl_path) or ".", exist_ok=True)
    with open(out_jsonl_path, "wb") as f:
        f.write(blob)

    return out_jsonl_path


def load_manifest_index(manifest_path: str) -> Dict[str, Dict[str, str]]:
    """
    key -> metadata (file, paragraph, feedback_segment, annotation_comment)
    """
    idx: Dict[str, Dict[str, str]] = {}
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = obj.get("key")
            if not isinstance(key, str) or not key:
                raise ValueError(f"Manifest line {line_no}: missing/invalid key")
            idx[key] = obj
    return idx


def load_batch_output_scores(
    *,
    results_jsonl_path: str,
) -> Dict[str, Dict[str, Union[str, int]]]:
    """
    Returns mapping: key -> { "raw_text": str, "score": int }
    """
    out: Dict[str, Dict[str, Union[str, int]]] = {}

    with open(results_jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            resp = json.loads(line)
            text = _first_text_from_generate_content_response(resp)
            score = _extract_score_0_to_4(text)

            out[str(line_no)] = {"raw_text": text, "score": score}

    return out


def make_batched_gemini_similarity_score(
    *,
    manifest_path: str,
    results_jsonl_path: str,
    strict: bool = True,
) -> Callable[[str, str, bool, bool], Union[int, str]]:
    """
    Returns a function with the SAME signature as gemini_similarity_score,
    but it reads from downloaded batch outputs instead of calling the API.

    - Uses deterministic key = sha256(feedback1 + \\0 + feedback2)
    - Looks up expected pair in manifest (sanity check)
    - Then finds the matching output by scanning the outputs and matching key->line via an index file we build once.

    To avoid O(N) scan per call, we build a key->score index once here.
    """

    manifest = load_manifest_index(manifest_path)

    outputs_by_line = []
    with open(results_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                outputs_by_line.append(json.loads(line))

    # Re-read manifest in order to preserve request order
    ordered_keys = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            ordered_keys.append(obj["key"])

    if len(outputs_by_line) != len(ordered_keys):
        raise RuntimeError(
            f"Mismatch: results lines={len(outputs_by_line)} vs manifest lines={len(ordered_keys)}. "
            "Cannot safely align outputs to requests."
        )

    key_to_output: Dict[str, Dict[str, Union[str, int]]] = {}
    for key, resp in zip(ordered_keys, outputs_by_line):
        raw_text = _first_text_from_generate_content_response(resp)
        score = _extract_score_0_to_4(raw_text)
        key_to_output[key] = {"raw_text": raw_text, "score": score}

    def batched_gemini_similarity_score(
        feedback1: str,
        feedback2: str,
        printApiLog: bool = False,
        scoreOnly: bool = True,
    ) -> Union[int, str]:
        key = _stable_pair_key(feedback1, feedback2)

        if key not in manifest:
            if strict:
                raise KeyError("Pair key not found in manifest (did Stage 1 generate this pair?)")
            return 0 if scoreOnly else ""

        m = manifest[key]
        if strict:
            if str(m.get("feedback_segment", "")) != str(feedback1):
                raise ValueError("Sanity check failed: feedback1 differs from manifest for this key.")
            if str(m.get("annotation_comment", "")) != str(feedback2):
                raise ValueError("Sanity check failed: feedback2 differs from manifest for this key.")

        out = key_to_output.get(key)
        if out is None:
            if strict:
                raise KeyError("Key exists in manifest but no output found in results alignment.")
            return 0 if scoreOnly else ""

        if not scoreOnly:
            return str(out["raw_text"] or "")

        score = int(out["score"])
        if printApiLog:
            print("BATCH GEMINI SIM:", str(feedback1)[:20], "---", str(feedback2)[:20], "=>", score)
        return score if score in range(5) else 0

    return batched_gemini_similarity_score


In [ ]:
results_path = poll_and_download_batch_results(
    client=gem_client,
    job_name="batches/jobID",   # <-- from Stage 1 output
    out_jsonl_path="./unimatch_batches/eval_"+setting+"/batch_results.jsonl",
    poll_seconds=10,
)

In [ ]:
# 3) run unimatch
unimatch(
    model_source='./segment_individual/seg_'+setting,
    annot_source='./segment_individual/seg_annot/',
    dest_dir='./unimatch_individual/eval_'+setting,
    match_threshold=2,
    num_files=None,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True,
)

#### Stage 2 (Download results and calculate metrics)

In [ ]:
def _extract_score_0_to_4(text: str) -> int:
    """Extract a single integer 0-4. Returns 0 if invalid."""
    if not text:
        return 0
    s = text.strip()
    if re.fullmatch(r"[0-4]", s):
        return int(s)
    m = re.search(r"(?<!\d)([0-4])(?!\d)", s)
    return int(m.group(1)) if m else 0


def _stable_pair_key(feedback1: str, feedback2: str) -> str:
    """Deterministic key = sha256(feedback1 + \\0 + feedback2). Must match Stage 1."""
    f1 = "" if feedback1 is None else str(feedback1)
    f2 = "" if feedback2 is None else str(feedback2)
    return hashlib.sha256((f1 + "\u0000" + f2).encode("utf-8")).hexdigest()


def _first_text_from_batch_line(obj: Dict[str, Any]) -> str:
    """
    Batch output lines:
      {"key": "...", "response": { ...GenerateContentResponse... }, ...}

    We extract candidates[0].content.parts[*].text from obj["response"].
    """
    resp = obj.get("response") if isinstance(obj.get("response"), dict) else obj

    cands = resp.get("candidates") or []
    if not cands:
        return ""
    content = cands[0].get("content") or {}
    parts = content.get("parts") or []
    for p in parts:
        t = p.get("text")
        if isinstance(t, str) and t.strip():
            return t
    return ""


def poll_and_download_batch_results(
    *,
    client: genai.Client,
    job_name: str,
    out_jsonl_path: str,
    poll_seconds: int = 10,
) -> str:
    """
    Poll job state until terminal, then download the results file to out_jsonl_path.
    """
    completed = {
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED",
    }

    batch_job = client.batches.get(name=job_name)
    while batch_job.state.name not in completed:
        time.sleep(poll_seconds)
        batch_job = client.batches.get(name=job_name)

    if batch_job.state.name != "JOB_STATE_SUCCEEDED":
        raise RuntimeError(f"Batch job did not succeed: state={batch_job.state.name}, error={batch_job.error}")

    if not (batch_job.dest and batch_job.dest.file_name):
        raise RuntimeError("Succeeded job has no dest.file_name (no file results found).")

    result_file_name = batch_job.dest.file_name
    blob = client.files.download(file=result_file_name)  # bytes

    os.makedirs(os.path.dirname(out_jsonl_path) or ".", exist_ok=True)
    with open(out_jsonl_path, "wb") as f:
        f.write(blob)

    return out_jsonl_path


def load_manifest_index(manifest_path: str) -> Dict[str, Dict[str, Any]]:
    """key -> manifest record (includes exact ms/ac texts)."""
    idx: Dict[str, Dict[str, Any]] = {}
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = obj.get("key")
            if not isinstance(key, str) or not key:
                raise ValueError(f"Manifest line {line_no}: missing/invalid key")
            idx[key] = obj
    return idx


def load_batch_results_by_key(results_jsonl_path: str) -> Dict[str, Dict[str, Union[str, int]]]:
    """
    key -> {raw_text, score}
    Uses the 'key' present in each batch output line.
    """
    key_to_output: Dict[str, Dict[str, Union[str, int]]] = {}
    with open(results_jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            key = obj.get("key")
            if not isinstance(key, str) or not key:
                # If this happens, your batch output format differs; stop early.
                raise ValueError(f"Results line {line_no}: missing 'key' field.")

            raw_text = _first_text_from_batch_line(obj)
            score = _extract_score_0_to_4(raw_text)
            key_to_output[key] = {"raw_text": raw_text, "score": score}

    if not key_to_output:
        raise RuntimeError("No outputs loaded from results JSONL.")
    return key_to_output


def make_batched_gemini_similarity_score(
    *,
    manifest_path: str,
    results_jsonl_path: str,
    strict: bool = True,
) -> Callable[[str, str, bool, bool], Union[int, str]]:
    """
    Returns a drop-in replacement for gemini_similarity_score(feedback1, feedback2, printApiLog, scoreOnly).
    It never calls the API; it reads from batch results.

    strict=True:
      - requires the pair key exist in the manifest
      - verifies feedback1/feedback2 exactly match manifest text for that key
    """
    manifest = load_manifest_index(manifest_path)
    key_to_output = load_batch_results_by_key(results_jsonl_path)

    def batched_gemini_similarity_score(
        feedback1: str,
        feedback2: str,
        printApiLog: bool = False,
        scoreOnly: bool = True,
    ) -> Union[int, str]:
        key = _stable_pair_key(feedback1, feedback2)

        if key not in manifest:
            if strict:
                raise KeyError("Pair key not found in manifest (did Stage 1 generate this pair?)")
            return 0 if scoreOnly else ""

        m = manifest[key]
        if strict:
            if str(m.get("feedback_segment", "")) != str(feedback1):
                raise ValueError("Sanity check failed: feedback1 differs from manifest for this key.")
            if str(m.get("annotation_comment", "")) != str(feedback2):
                raise ValueError("Sanity check failed: feedback2 differs from manifest for this key.")

        out = key_to_output.get(key)
        if out is None:
            if strict:
                raise KeyError("Key exists in manifest but no output found in results JSONL.")
            return 0 if scoreOnly else ""

        raw_text = str(out.get("raw_text") or "")
        score = int(out.get("score") or 0)

        if printApiLog:
            print("BATCH GEMINI SIM:", str(feedback1)[:20], "---", str(feedback2)[:20], "=>", score, "| raw:", raw_text)

        if not scoreOnly:
            return raw_text

        return score if score in range(5) else 0

    return batched_gemini_similarity_score

gemini_similarity_score = make_batched_gemini_similarity_score(
    manifest_path="./unimatch_batches/eval_"+setting+"/batch_manifest.jsonl",
    results_jsonl_path=results_path,
    strict=True,
)


In [ ]:

setting = 's3'

unimatch(
    model_source= './segment_individual/seg_'+setting,
    annot_source= './segment_individual/seg_annot/',
    dest_dir= './unimatch_individual/eval_'+setting,
    match_threshold = 2,
    num_files = 5,
    continue_ind = None,
    printApiLog = False,
    sub56 = False,
    one_GT_only = True
)

### GPT Similarity (Batch API)

####  Stage 1 (Creating and submitting a batch)

In [ ]:
import os
import json
import hashlib
from typing import Optional, Dict
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_KEY"])

MODEL_NAME = "gpt-5.1"


def _stable_pair_key(feedback1: str, feedback2: str) -> str:
    f1 = "" if feedback1 is None else str(feedback1)
    f2 = "" if feedback2 is None else str(feedback2)
    return hashlib.sha256((f1 + "\u0000" + f2).encode("utf-8")).hexdigest()


def build_and_submit_openai_batch(
    *,
    model_source: str,
    annot_source: str,
    out_dir: str,
    prompt_template: str,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    sub56: bool = False,
    one_GT_only: bool = False,
) -> Dict[str, str]:

    os.makedirs(out_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    sub56_names = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    batch_path = os.path.join(out_dir, "openai_batch_requests.jsonl")
    manifest_path = os.path.join(out_dir, "openai_batch_manifest.jsonl")

    n_written = 0

    with open(batch_path, "w", encoding="utf-8") as batch_fp, \
         open(manifest_path, "w", encoding="utf-8") as man_fp:

        for fname in selected:

            if sub56 and fname not in sub56_names:
                continue

            model_path = os.path.join(model_source, fname)
            annot_path = os.path.join(annot_source, fname)
            if not os.path.isfile(annot_path):
                continue

            with open(model_path, "r", encoding="utf-8") as f:
                model_data = json.load(f)
            with open(annot_path, "r", encoding="utf-8") as f:
                annot_data = json.load(f)

            for p_idx, (p_model, p_annot) in enumerate(
                zip(model_data.get("paragraphs", []),
                    annot_data.get("paragraphs", []))
            ):

                annots = p_annot.get("annotations", [])
                if one_GT_only and len(annots) != 1:
                    continue

                model_list = p_model.get("model")
                if not isinstance(model_list, list) or len(model_list) != 1:
                    continue

                feedback = model_list[0].get("feedback")
                if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                    continue

                seg_idx = feedback.index("FEEDBACK_SEGMENTS")
                model_segs = [
                    s for s in feedback[seg_idx + 1:]
                    if isinstance(s, str) and len(s.strip()) >= 3
                ]

                comments = [
                    a.get("comment", "").strip()
                    for a in annots
                    if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
                ]

                for ms in model_segs:
                    for ac in comments:

                        key = _stable_pair_key(ms, ac)

                        prompt = prompt_template.format(
                            feedback1=str(ms),
                            feedback2=str(ac)
                        )

                        batch_fp.write(json.dumps({
                            "custom_id": key,
                            "method": "POST",
                            "url": "/v1/responses",
                            "body": {
                                "model": MODEL_NAME,
                                "reasoning": {"effort": "low"},
                                #"max_output_tokens": 10,
                                "input": prompt
                            }
                        }) + "\n")

                        man_fp.write(json.dumps({
                            "custom_id": key,
                            "file": fname,
                            "paragraph": p_idx,
                            "feedback_segment": ms,
                            "annotation_comment": ac,
                        }) + "\n")

                        n_written += 1

    if n_written == 0:
        raise RuntimeError("No requests written.")

    # Upload file
    file_obj = client.files.create(
        file=open(batch_path, "rb"),
        purpose="batch"
    )

    # Create batch job
    job = client.batches.create(
        input_file_id=file_obj.id,
        endpoint="/v1/responses",
        completion_window="24h"
    )

    return {
        "job_id": job.id,
        "input_file_id": file_obj.id,
        "batch_path": batch_path,
        "manifest_path": manifest_path,
        "n_written": n_written
    }


In [ ]:
def poll_and_download_openai_batch(
    job_id: str,
    out_path: str,
    poll_seconds: int = 10
) -> str:

    while True:
        job = client.batches.retrieve(job_id)

        if job.status in ("completed", "failed", "cancelled"):
            break

        time.sleep(poll_seconds)

    if job.status != "completed":
        raise RuntimeError(f"Batch failed: {job.status}")

    result_file_id = job.output_file_id

    result = client.files.content(result_file_id)

    with open(out_path, "wb") as f:
        f.write(result.read())

    return out_path


#### Stage 2 (download results and calculate metrics)

In [ ]:
import re
import json


def _extract_score_0_to_4(text: str) -> int:
    if not text:
        return 0
    s = text.strip()
    if re.fullmatch(r"[0-4]", s):
        return int(s)
    m = re.search(r"(?<!\d)([0-4])(?!\d)", s)
    return int(m.group(1)) if m else 0


def make_batched_openai_similarity_score(
    *,
    results_jsonl_path: str,
    strict: bool = True,
):

    results = {}

    with open(results_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            key = obj["custom_id"]

            # Extract text from OpenAI Responses API output
            output_text = ""
            try:
                output_text = obj["response"]["body"]["output"][1]["content"][0]["text"]
            except Exception:
                output_text = ""

            score = _extract_score_0_to_4(output_text)

            results[key] = {
                "raw_text": output_text,
                "score": score
            }

    def openai_batched_similarity_score(
        feedback1: str,
        feedback2: str,
        printApiLog: bool = False,
        scoreOnly: bool = True,
    ):

        key = _stable_pair_key(feedback1, feedback2)

        out = results.get(key)
        if out is None:
            if strict:
                raise KeyError("Key not found in batch results.")
            return 0 if scoreOnly else ""

        if printApiLog:
            print("OPENAI BATCH:", key[:8], "=>", out["score"])

        if not scoreOnly:
            return out["raw_text"]

        return out["score"]

    return openai_batched_similarity_score


In [ ]:
import os
import json
from typing import Optional, Dict, List




def unimatch(
    model_source: str,
    annot_source: str,
    dest_dir: str,
    results_jsonl_path: str,
    match_threshold: int = 2,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
):
    offline_gpt_similarity_score = make_batched_openai_similarity_score(
    results_jsonl_path=results_jsonl_path,
    strict=True
    )
    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")

    os.makedirs(dest_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))

    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    micro_TP = micro_FP = micro_FN = 0
    macro_precisions = []
    macro_recalls = []
    macro_f1s = []

    for i, fname in enumerate(selected, 1):
        print(f"PROCESSING: {i}/{len(selected)} {fname}")

        if sub56 and fname not in sub56_names:
            print("SKIPPED...")
            continue

        model_path = os.path.join(model_source, fname)
        annot_path = os.path.join(annot_source, fname)
        dst_path = os.path.join(dest_dir, fname)

        if not os.path.isfile(annot_path):
            print(f"[ERROR] Missing annotation file: {fname}")
            continue

        with open(model_path, "r", encoding="utf-8") as f:
            model_data = json.load(f)
        with open(annot_path, "r", encoding="utf-8") as f:
            annot_data = json.load(f)

        for idx, (p_model, p_annot) in enumerate(
            zip(model_data.get("paragraphs", []),
                annot_data.get("paragraphs", []))
        ):
            model_list = p_model.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            feedback = model_list[0].get("feedback")
            if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                continue

            seg_idx = feedback.index("FEEDBACK_SEGMENTS")
            model_segs = [
                s for s in feedback[seg_idx + 1:]
                if isinstance(s, str) and len(s.strip()) >= 3
            ]
            if not model_segs:
                continue
            
            annots = p_annot.get("annotations", [])

            if one_GT_only and (not isinstance(annots, list) or len(annots) != 1):
                continue

            comments = [
                a.get("comment", "").strip()
                for a in p_annot.get("annotations", [])
                if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
            ]
            if not comments:
                continue

            # ---- Matching ----
            model_matched = [False] * len(model_segs)
            annot_matched = [False] * len(comments)
            segsim: List[Dict] = []

            for mi, ms in enumerate(model_segs):
                for ai, ac in enumerate(comments):
                    score = offline_gpt_similarity_score(ms, ac, printApiLog, scoreOnly=True)
                    segsim.append({
                        "feedback": ms,
                        "comment": ac,
                        "score": score,
                    })
                    if score >= match_threshold:
                        model_matched[mi] = True
                        annot_matched[ai] = True

            TP = sum(model_matched)
            FP = len(model_segs) - TP
            FN = annot_matched.count(False)

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

            print({
                "file": fname,
                "paragraph": idx,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            })

            micro_TP += TP
            micro_FP += FP
            micro_FN += FN

            macro_precisions.append(precision)
            macro_recalls.append(recall)
            macro_f1s.append(f1)

            p_model["segsim"] = segsim

        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(model_data, f, ensure_ascii=False, indent=2)

    # ---- Final metrics ----
    micro_precision = micro_TP / (micro_TP + micro_FP) if (micro_TP + micro_FP) > 0 else 0.0
    micro_recall = micro_TP / (micro_TP + micro_FN) if (micro_TP + micro_FN) > 0 else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0 else 0.0
    )

    print("FINAL METRICS")
    print({
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": sum(macro_precisions) / len(macro_precisions) if macro_precisions else 0.0,
        "macro_recall": sum(macro_recalls) / len(macro_recalls) if macro_recalls else 0.0,
        "macro_f1": sum(macro_f1s) / len(macro_f1s) if macro_f1s else 0.0,
    })


#### Running Stage 1 and 2 sample

In [ ]:
setting = "s3"

out_dir = f"./openai_batches/eval_{setting}"

job_info = build_and_submit_openai_batch(
    model_source=f"./segment_individual/seg_{setting}",
    annot_source="./segment_individual/seg_annot/",
    out_dir=out_dir,
    prompt_template=GPT_SIM_PROMPT,
    num_files=5,
    continue_ind=None,
    sub56=False,
    one_GT_only=True,
)

print("Batch submitted.")
print("Job ID:", job_info["job_id"])
print("Requests written:", job_info["n_written"])


In [ ]:
# Check Status

job_id = job_info["job_id"] 

job = client.batches.retrieve(job_id)

print("Status:", job.status)

if job.status == "completed":
    print(job_id)
    print("Output file id:", job.output_file_id)

job_id = job_info["job_id"]
job = client.batches.retrieve(job_id)

print("Status:", job.status)
print("Request counts:", getattr(job, "request_counts", None))
print("Output file id:", getattr(job, "output_file_id", None))
print("Error file id:", getattr(job, "error_file_id", None))


In [ ]:
err_id = job.error_file_id
if err_id:
    err = client.files.content(err_id)
    with open("batch_errors.jsonl", "wb") as f:
        f.write(err.read())
    print("Downloaded errors to batch_errors.jsonl")
else:
    print("No error_file_id on the job.")

In [ ]:
results_path = f"./openai_batches/eval_{setting}/openai_results.jsonl"

results_path = poll_and_download_openai_batch(
    job_id=job_id,
    out_path=results_path,
    poll_seconds=10
)

print("Downloaded to:", results_path)


In [ ]:
unimatch(
    model_source=f'./segment_individual/seg_{setting}',
    annot_source='./segment_individual/seg_annot/',
    dest_dir=f'./unimatch_individual/eval_{setting}',
    results_jsonl_path = results_path,
    match_threshold=3,
    num_files=5,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True
)


### Buffering Approach (Two Stage)

In [ ]:
import re
import time
from collections import deque
from typing import Deque, List, Optional, Tuple


### PROMPT AND CLIENT MUST BE READY

def _make_tags(n: int) -> List[str]:
    # A, B, ..., Z, AA, AB, ...
    tags: List[str] = []
    i = 0
    while len(tags) < n:
        x = i
        s = ""
        while True:
            s = chr(ord("A") + (x % 26)) + s
            x = x // 26 - 1
            if x < 0:
                break
        tags.append(s)
        i += 1
    return tags


def _build_pairs_block(pairs: List[Tuple[str, str]]) -> str:
    tags = _make_tags(len(pairs))
    chunks: List[str] = []
    for tag, (f1, f2) in zip(tags, pairs):
        chunks.append(
            f"{tag})\n"
            f"Instructor A feedback: <feedback>{f1}</feedback>\n"
            f"Instructor B feedback: <feedback>{f2}</feedback>\n"
        )
    return "\n".join(chunks)


def _parse_tagged_scores(text: str, expected_tags: List[str]) -> Optional[List[int]]:
    # STRICT: reject duplicates, missing tags, extra tags.
    scores_by_tag: dict[str, int] = {}

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        m = re.match(r"^([A-Z]{1,10})\)\s*([0-4])\s*$", line)
        if not m:
            continue

        tag, score_str = m.group(1), m.group(2)

        # reject duplicates
        if tag in scores_by_tag:
            return None

        scores_by_tag[tag] = int(score_str)

    if len(scores_by_tag) != len(expected_tags):
        return None

    if set(scores_by_tag.keys()) != set(expected_tags):
        return None

    return [scores_by_tag[tag] for tag in expected_tags]


class GPTSimilarityBuffered:
    """
    Buffers (feedback1, feedback2) pairs; sends to GPT when buffer hits buffer_size
    or when flush=True is passed.
    """

    def __init__(
        self,
        client,
        buffer_size: int,
        model: str = "gpt-5-nano",
        printApiLog: bool = False,
        max_attempts: int = 5,
        max_parse_retries: int = 3,
    ):
        if buffer_size <= 0:
            raise ValueError("buffer_size must be >= 1")
        if max_attempts <= 0:
            raise ValueError("max_attempts must be >= 1")
        if max_parse_retries <= 0:
            raise ValueError("max_parse_retries must be >= 1")

        self.client = client
        self.buffer_size = buffer_size
        self.model = model
        self.printApiLog = printApiLog
        self.max_attempts = max_attempts
        self.max_parse_retries = max_parse_retries

        self._buffer: List[Tuple[str, str]] = []
        self._pending: Deque[int] = deque()

    def add_pair(
        self,
        feedback1: Optional[str],
        feedback2: Optional[str],
        flush: bool = False
    ) -> List[int]:
        produced: List[int] = []

        # Drain any pending scores first
        while self._pending:
            produced.append(self._pending.popleft())

        # Add current pair unless this call is flush-only
        if feedback1 is not None and feedback2 is not None:
            self._buffer.append((feedback1, feedback2))

        should_send = flush or (len(self._buffer) >= self.buffer_size)
        if not should_send:
            if self.printApiLog:
                print(feedback1[:10], '----', feedback2[:10], " Buffered...")
            return produced  # likely empty

        # If flush=True and buffer is empty, nothing to do
        if not self._buffer:
            return produced

        batch_pairs = self._buffer
        self._buffer = []

        tags = _make_tags(len(batch_pairs))
        pairs_block = _build_pairs_block(batch_pairs)
        prompt_text = GPT_SIM_BATCH_PROMPT.format(pairs_block=pairs_block)

        if self.printApiLog:
            print(f"GPT Model: {self.model} | batch={len(batch_pairs)}")

        parsed: Optional[List[int]] = None
        parse_tries = 0

        for attempt in range(self.max_attempts):
            try:
                response = self.client.responses.create(
                    model=self.model,
                    #model="gpt-5.1",
                    reasoning={"effort": "low"},
                    #temperature=0.1,
                    #max_output_tokens=10,
                    #top_p=1,
                    input=[{"role": "user", "content": prompt_text}],
                )
                out_text = response.output_text.strip()

                parsed = _parse_tagged_scores(out_text, tags)
                if parsed is None:
                    parse_tries += 1
                    if self.printApiLog:
                        preview = out_text.replace("\n", " ")[:200]
                        print(
                            f"Bad parse (parse_try {parse_tries}/{self.max_parse_retries}) | {preview}"
                        )

                    # Retry parse by re-calling the model
                    if parse_tries < self.max_parse_retries:
                        time.sleep(1)
                        continue

                    # Too many bad parses -> fallback zeros
                    parsed = [0] * len(batch_pairs)

                # Valid parsed (or fallback zeros)
                for s in parsed:
                    self._pending.append(s)
                break

            except Exception as e:
                if self.printApiLog:
                    print(f"Attempt {attempt + 1} failed:", e)
                time.sleep(2)

        if parsed is None:
            # All API attempts failed
            for _ in batch_pairs:
                self._pending.append(0)

        while self._pending:
            produced.append(self._pending.popleft())

        return produced


In [ ]:
import os
import json
from collections import deque
from typing import Optional, Dict, List, Tuple, Deque, Any

def stage1_score_and_save_pairs(
    *,
    model_source: str,
    annot_source: str,
    scores_jsonl_path: str,
    scorer,  # GPTSimilarityBuffered instance
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
    g1_GT_only: bool = False,
) -> Dict[str, int]:
    """
    Iterates like unimatch, but instead of instant API calls it uses the provided
    buffered scorer (GPTSimilarityBuffered) and writes ONE JSONL record per pair.

    JSONL: file, paragraph_index, model_seg_index, annot_index, feedback, comment, score

    Returns counts for sanity checking.
    """
    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")
    
    if g1_GT_only and one_GT_only:
        raise ValueError("Can't set both one_GT_only and g1_GT_only to True.")
    
    if g1_GT_only:
        scores_jsonl_path = scores_jsonl_path[:-6]+"_g1GT.jsonl"

    out_dir = os.path.dirname(os.path.abspath(scores_jsonl_path)) or "."
    os.makedirs(out_dir, exist_ok=True)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    sub56_names: Optional[set[str]] = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    # Queue of metadata in the exact order pairs are added to scorer
    pending_meta: Deque[Dict[str, Any]] = deque()

    total_files_seen = 0
    total_files_scored = 0
    total_pairs_added = 0
    total_pairs_written = 0

    def _write_scored_items(fh, scores: List[int]) -> None:
        nonlocal total_pairs_written
        for s in scores:
            if not pending_meta:
                raise RuntimeError(
                    "Internal mismatch: got a score from scorer but no pending metadata. "
                    "This indicates a logic error in buffering alignment."
                )
            meta = pending_meta.popleft()
            meta["score"] = int(s)
            fh.write(json.dumps(meta, ensure_ascii=False) + "\n")
            total_pairs_written += 1

    with open(scores_jsonl_path, "w", encoding="utf-8") as out_f:
        for i, fname in enumerate(selected, 1):
            total_files_seen += 1
            if printApiLog:
                print(f"STAGE1 PROCESSING: {i}/{len(selected)} {fname}")

            if sub56 and sub56_names is not None and fname not in sub56_names:
                if printApiLog:
                    print("STAGE1 SKIPPED (sub56)...")
                continue

            model_path = os.path.join(model_source, fname)
            annot_path = os.path.join(annot_source, fname)

            if not os.path.isfile(annot_path):
                if printApiLog:
                    print(f"[STAGE1 ERROR] Missing annotation file: {fname}")
                continue

            with open(model_path, "r", encoding="utf-8") as f:
                model_data = json.load(f)
            with open(annot_path, "r", encoding="utf-8") as f:
                annot_data = json.load(f)

            wrote_any_for_file = False

            for p_idx, (p_model, p_annot) in enumerate(
                zip(model_data.get("paragraphs", []), annot_data.get("paragraphs", []))
            ):
                model_list = p_model.get("model")
                if not isinstance(model_list, list) or len(model_list) != 1:
                    continue

                feedback = model_list[0].get("feedback")
                if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                    continue

                seg_idx = feedback.index("FEEDBACK_SEGMENTS")
                model_segs = [
                    s for s in feedback[seg_idx + 1:]
                    if isinstance(s, str) and len(s.strip()) >= 3
                ]
                if not model_segs:
                    continue

                comments = [
                    a.get("comment", "").strip()
                    for a in p_annot.get("annotations", [])
                    if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
                ]
                if not comments:
                    continue

                if one_GT_only and len(comments) != 1:
                    continue
                
                if g1_GT_only and  len(comments) == 1:
                    continue

                # Add ALL pairs (mi, ai) to scorer, record metadata in same order
                for mi, ms in enumerate(model_segs):
                    for ai, ac in enumerate(comments):
                        pending_meta.append({
                            "file": fname,
                            "paragraph_index": p_idx,
                            "model_seg_index": mi,
                            "annot_index": ai,
                            "feedback": ms,
                            "comment": ac,
                        })
                        total_pairs_added += 1
                        wrote_any_for_file = True

                        produced = scorer.add_pair(ms, ac, flush=False)
                        if produced:
                            _write_scored_items(out_f, produced)

            if wrote_any_for_file:
                total_files_scored += 1

        # flush leftovers
        produced = scorer.add_pair(None, None, flush=True)
        if produced:
            _write_scored_items(out_f, produced)

    if pending_meta:
        raise RuntimeError(
            f"Stage1 ended but still has {len(pending_meta)} pending metadata items. "
            "This indicates a mismatch between pairs added and scores produced."
        )

    return {
        "files_seen": total_files_seen,
        "files_scored": total_files_scored,
        "pairs_added": total_pairs_added,
        "pairs_written": total_pairs_written,
    }

# Stage 2: unimatch using saved scores 


def _load_scores_jsonl_to_lookup(scores_jsonl_path: str) -> Dict[Tuple[str, int, int, int], int]:
    """
    Loads JSONL produced by stage1 into a lookup:
      key = (file, paragraph_index, model_seg_index, annot_index) -> score
    Raises on duplicate keys.
    """
    if not os.path.isfile(scores_jsonl_path):
        raise FileNotFoundError(f"scores_jsonl_path not found: {scores_jsonl_path}")

    lookup: Dict[Tuple[str, int, int, int], int] = {}
    with open(scores_jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no} in {scores_jsonl_path}: {e}") from e

            for field in ["file", "paragraph_index", "model_seg_index", "annot_index", "score"]:
                if field not in obj:
                    raise ValueError(f"Missing '{field}' on line {line_no} in {scores_jsonl_path}")

            key = (
                str(obj["file"]),
                int(obj["paragraph_index"]),
                int(obj["model_seg_index"]),
                int(obj["annot_index"]),
            )
            if key in lookup:
                raise ValueError(f"Duplicate key in scores file on line {line_no}: {key}")

            lookup[key] = int(obj["score"])

    return lookup


def unimatch_from_saved_scores(
    *,
    model_source: str,
    annot_source: str,
    dest_dir: str,
    scores_jsonl_path: str,
    match_threshold: int = 2,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
    strict_missing: bool = True,
) -> Dict[str, float]:
    """
    Same as unimatch, but uses precomputed scores from stage1 JSONL.
    """
    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")

    os.makedirs(dest_dir, exist_ok=True)

    scores_lookup = _load_scores_jsonl_to_lookup(scores_jsonl_path)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    sub56_names: Optional[set[str]] = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    micro_TP = micro_FP = micro_FN = 0
    macro_precisions: List[float] = []
    macro_recalls: List[float] = []
    macro_f1s: List[float] = []
    total_llm_segments = 0

    for i, fname in enumerate(selected, 1):
        if printApiLog:
            print(f"PROCESSING: {i}/{len(selected)} {fname}")

        if sub56 and sub56_names is not None and fname not in sub56_names:
            if printApiLog:
                print("SKIPPED...")
            continue

        model_path = os.path.join(model_source, fname)
        annot_path = os.path.join(annot_source, fname)
        dst_path = os.path.join(dest_dir, fname)

        if not os.path.isfile(annot_path):
            print(f"[ERROR] Missing annotation file: {fname}")
            continue

        with open(model_path, "r", encoding="utf-8") as f:
            model_data = json.load(f)
        with open(annot_path, "r", encoding="utf-8") as f:
            annot_data = json.load(f)

        for p_idx, (p_model, p_annot) in enumerate(
            zip(model_data.get("paragraphs", []), annot_data.get("paragraphs", []))
        ):
            model_list = p_model.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            feedback = model_list[0].get("feedback")
            if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                continue

            seg_idx = feedback.index("FEEDBACK_SEGMENTS")
            model_segs = [
                s for s in feedback[seg_idx + 1:]
                if isinstance(s, str) and len(s.strip()) >= 3
            ]
            if not model_segs:
                continue
            

            comments = [
                a.get("comment", "").strip()
                for a in p_annot.get("annotations", [])
                if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
            ]
            if not comments:
                continue

            if one_GT_only and len(comments) != 1:
                continue
            
            total_llm_segments += len(model_segs)

            model_matched = [False] * len(model_segs)
            annot_matched = [False] * len(comments)
            segsim: List[Dict[str, Any]] = []

            for mi, ms in enumerate(model_segs):
                for ai, ac in enumerate(comments):
                    key = (fname, p_idx, mi, ai)
                    if key not in scores_lookup:
                        if strict_missing:
                            raise KeyError(
                                f"Missing score for pair {key} in {scores_jsonl_path}. "
                                "If you want missing scores to default to 0, set strict_missing=False."
                            )
                        score = 0
                    else:
                        score = int(scores_lookup[key])

                    segsim.append({
                        "feedback": ms,
                        "comment": ac,
                        "score": score,
                    })
                    if score >= match_threshold:
                        model_matched[mi] = True
                        annot_matched[ai] = True

            TP = sum(model_matched)
            FP = len(model_segs) - TP
            FN = annot_matched.count(False)

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

            if printApiLog:
                print({
                    "file": fname,
                    "paragraph": p_idx,
                    "TP": TP,
                    "FP": FP,
                    "FN": FN,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                })

            micro_TP += TP
            micro_FP += FP
            micro_FN += FN

            macro_precisions.append(precision)
            macro_recalls.append(recall)
            macro_f1s.append(f1)

            p_model["segsim"] = segsim

        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(model_data, f, ensure_ascii=False, indent=2)

    micro_precision = micro_TP / (micro_TP + micro_FP) if (micro_TP + micro_FP) > 0 else 0.0
    micro_recall = micro_TP / (micro_TP + micro_FN) if (micro_TP + micro_FN) > 0 else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0 else 0.0
    )
    
    
    metrics = {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": sum(macro_precisions) / len(macro_precisions) if macro_precisions else 0.0,
        "macro_recall": sum(macro_recalls) / len(macro_recalls) if macro_recalls else 0.0,
        "macro_f1": sum(macro_f1s) / len(macro_f1s) if macro_f1s else 0.0,
        "total_llm_segments": total_llm_segments
    }

    if printApiLog:
        print("\n=== FINAL METRICS ===")
        print(metrics)

    return metrics


#### Buffering Sample Usage

In [ ]:
setting='s1'

# Define Scorer (GPT or Gemini)
# Gemini:
scorer = GeminiSimilarityBuffered(
    gem_client=gem_client,
    buffer_size=50,
    model="gemini-3.1-flash-lite-preview",
    prompt_template=GEMINI_SIM_BATCH_PROMPT,
    printApiLog=True,
)

# GPT:
#scorer = GPTSimilarityBuffered(client=client, buffer_size=50, model="gpt-5-mini-2025-08-07", printApiLog=True)


# Stage 1
stats = stage1_score_and_save_pairs(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    scorer=scorer,
    num_files=5,
    continue_ind=None,
    printApiLog=True,
    sub56=False,
    one_GT_only=False,
    g1_GT_only=True,
)


In [ ]:
# Sanity check with GPT which was done before:

In [ ]:

scorer = GPTSimilarityBuffered(client=client, buffer_size=50, model="gpt-5-mini-2025-08-07", printApiLog=True)

stats = stage1_score_and_save_pairs(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    scorer=scorer,
    num_files=5,
    continue_ind=None,
    printApiLog=True,
    sub56=False,
    one_GT_only=True,
)

#### 1-1 matching (Threshold Based)

In [ ]:
import os
import json
from typing import Optional, Dict, List, Any
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching


def unimatch_from_saved_scores_1to1(
    *,
    model_source: str,
    annot_source: str,
    dest_dir: str,
    scores_jsonl_path: str,
    match_threshold: int = 2,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
    strict_missing: bool = True,
) -> Dict[str, float]:
    """
    Same as unimatch_from_saved_scores, but uses maximum 1-1 bipartite matching
    between model segments and GT comments.

    strict_missing=True:
      - raise error if any required pair score is missing
    strict_missing=False:
      - missing scores default to 0
    """

    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")

    os.makedirs(dest_dir, exist_ok=True)

    scores_lookup = _load_scores_jsonl_to_lookup(scores_jsonl_path)
    # Optional fallback lookup (scores_jsonl_path with "_g1GT.jsonl")
    scores_lookup_g1GT = None
    g1gt_path = scores_jsonl_path[:-6] + "_g1GT.jsonl"

    if os.path.isfile(g1gt_path):
        scores_lookup_g1GT = _load_scores_jsonl_to_lookup(g1gt_path)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    sub56_names: Optional[set[str]] = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    micro_TP = micro_FP = micro_FN = 0
    macro_precisions: List[float] = []
    macro_recalls: List[float] = []
    macro_f1s: List[float] = []
    total_llm_segments = 0

    for i, fname in enumerate(selected, 1):
        if printApiLog:
            print(f"PROCESSING: {i}/{len(selected)} {fname}")

        if sub56 and sub56_names is not None and fname not in sub56_names:
            if printApiLog:
                print("SKIPPED...")
            continue

        model_path = os.path.join(model_source, fname)
        annot_path = os.path.join(annot_source, fname)
        dst_path = os.path.join(dest_dir, fname)

        if not os.path.isfile(annot_path):
            print(f"[ERROR] Missing annotation file: {fname}")
            continue

        with open(model_path, "r", encoding="utf-8") as f:
            model_data = json.load(f)
        with open(annot_path, "r", encoding="utf-8") as f:
            annot_data = json.load(f)

        for p_idx, (p_model, p_annot) in enumerate(
            zip(model_data.get("paragraphs", []), annot_data.get("paragraphs", []))
        ):
            model_list = p_model.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            feedback = model_list[0].get("feedback")
            if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                continue

            seg_idx = feedback.index("FEEDBACK_SEGMENTS")
            model_segs = [
                s for s in feedback[seg_idx + 1:]
                if isinstance(s, str) and len(s.strip()) >= 3
            ]
            if not model_segs:
                continue

            comments = [
                a.get("comment", "").strip()
                for a in p_annot.get("annotations", [])
                if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
            ]
            if not comments:
                continue

            if one_GT_only and len(comments) != 1:
                continue

            total_llm_segments += len(model_segs)

            n_model = len(model_segs)
            n_annot = len(comments)

            segsim: List[Dict[str, Any]] = []
            adj = [[0] * n_annot for _ in range(n_model)]

            # Build adjacency matrix
            for mi, ms in enumerate(model_segs):
                for ai, ac in enumerate(comments):
                    key = (fname, p_idx, mi, ai)

                    if key in scores_lookup:
                        score = int(scores_lookup[key])

                    elif scores_lookup_g1GT is not None and key in scores_lookup_g1GT:
                        score = int(scores_lookup_g1GT[key])

                    else:
                        if strict_missing:
                            raise KeyError(
                                f"Missing score for pair {key} in both {scores_jsonl_path} "
                                f"and {g1gt_path}."
                            )
                        score = 0

                    segsim.append({
                        "feedback": ms,
                        "comment": ac,
                        "score": score,
                    })

                    if score >= match_threshold:
                        adj[mi][ai] = 1

            # Maximum 1-1 bipartite matching
            graph = csr_matrix(adj)
            matching = maximum_bipartite_matching(graph, perm_type='row')

            TP = sum(1 for m in matching if m != -1)
            FP = n_model - TP
            FN = n_annot - TP

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

            if printApiLog:
                print({
                    "file": fname,
                    "paragraph": p_idx,
                    "TP": TP,
                    "FP": FP,
                    "FN": FN,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                })

            micro_TP += TP
            micro_FP += FP
            micro_FN += FN

            macro_precisions.append(precision)
            macro_recalls.append(recall)
            macro_f1s.append(f1)

            p_model["segsim"] = segsim

        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(model_data, f, ensure_ascii=False, indent=2)

    micro_precision = micro_TP / (micro_TP + micro_FP) if (micro_TP + micro_FP) > 0 else 0.0
    micro_recall = micro_TP / (micro_TP + micro_FN) if (micro_TP + micro_FN) > 0 else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0 else 0.0
    )

    metrics = {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": sum(macro_precisions) / len(macro_precisions) if macro_precisions else 0.0,
        "macro_recall": sum(macro_recalls) / len(macro_recalls) if macro_recalls else 0.0,
        "macro_f1": sum(macro_f1s) / len(macro_f1s) if macro_f1s else 0.0,
        "total_llm_segments": total_llm_segments
    }

    if printApiLog:
        print("\n=== FINAL METRICS ===")
        print(metrics)

    return metrics


In [ ]:

setting='s1'

stats = stage1_score_and_save_pairs(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    scorer=scorer,
    num_files=None,
    continue_ind=None,
    printApiLog=True,
    sub56=False,
    one_GT_only=True,
)

print("STAGE1 STATS:", stats)


In [ ]:
# Sample Usage (Assuming First stage is ran. Whether GPT or Gemini, doesn't matter)

setting = 's1'

unimatch_from_saved_scores_1to1(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    dest_dir="./unimatch_individual/eval_"+setting,
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    match_threshold=3,
    num_files=None,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True,
    strict_missing=True, 
)

In [ ]:
setting = 's1'

unimatch_from_saved_scores(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    dest_dir="./unimatch_individual/eval_"+setting,
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    match_threshold=3,
    num_files=None,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True,
    strict_missing=True,
)

In [ ]:
setting = 's38'

unimatch_from_saved_scores(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    dest_dir="./unimatch_individual/eval_"+setting,
    scores_jsonl_path="./unimatch_individual/stage1_scores_"+setting+".jsonl",
    match_threshold=3,
    num_files=None,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True,
    strict_missing=True, 
)

#### CEAF 1-1 Matching (Threshold-free soft metrics)

In [ ]:
import os
import json
from typing import Optional, Dict, List, Any, Tuple
import numpy as np
from scipy.optimize import linear_sum_assignment


def _load_scores_jsonl_to_lookup(scores_jsonl_path: str) -> Dict[Tuple[str, int, int, int], int]:
    """
    Loads stage1 JSONL scores into a lookup:
      (file, paragraph_index, model_seg_index, annot_index) -> score
    """
    if not os.path.isfile(scores_jsonl_path):
        raise FileNotFoundError(f"scores_jsonl_path not found: {scores_jsonl_path}")

    lookup: Dict[Tuple[str, int, int, int], int] = {}

    with open(scores_jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                row = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON on line {line_num} of {scores_jsonl_path}: {e}"
                ) from e

            required = [
                "file",
                "paragraph_index",
                "model_seg_index",
                "annot_index",
                "score",
            ]
            missing = [k for k in required if k not in row]
            if missing:
                raise KeyError(
                    f"Missing required keys {missing} on line {line_num} of {scores_jsonl_path}"
                )

            key = (
                row["file"],
                int(row["paragraph_index"]),
                int(row["model_seg_index"]),
                int(row["annot_index"]),
            )
            score = int(row["score"])

            if key in lookup:
                raise ValueError(
                    f"Duplicate score entry for key {key} in {scores_jsonl_path} (line {line_num})"
                )

            lookup[key] = score

    return lookup


def unimatch_from_saved_scores_soft_1to1(
    *,
    model_source: str,
    annot_source: str,
    dest_dir: str,
    scores_jsonl_path: str,
    num_files: Optional[int] = None,
    continue_ind: Optional[int] = None,
    printApiLog: bool = False,
    sub56: bool = False,
    one_GT_only: bool = False,
    strict_missing: bool = True,
    g1_GT_only: bool = False,
) -> Dict[str, float]:
    """
    Threshold-free 1-to-1 evaluation using maximum-weight bipartite matching.

    For each paragraph:
      - build full score matrix from saved JSONL scores
      - find the maximum-sum 1-to-1 assignment
      - compute CEAF-style soft precision / recall / F1

    Denominator uses self-similarity = 4 for every unit, since the similarity
    scale is 0..4.

    strict_missing=True:
      - raise error if any required pair score is missing
    strict_missing=False:
      - missing scores default to 0
    """

    MAX_SELF_SIM = 4.0

    if not os.path.isdir(model_source):
        raise FileNotFoundError(f"model_source not found: {model_source}")
    if not os.path.isdir(annot_source):
        raise FileNotFoundError(f"annot_source not found: {annot_source}")

    os.makedirs(dest_dir, exist_ok=True)

    scores_lookup = _load_scores_jsonl_to_lookup(scores_jsonl_path)
    # Optional fallback lookup (scores_jsonl_path with "_g1GT.jsonl")
    scores_lookup_g1GT = None
    g1gt_path = scores_jsonl_path[:-6] + "_g1GT.jsonl"

    if os.path.isfile(g1gt_path):
        scores_lookup_g1GT = _load_scores_jsonl_to_lookup(g1gt_path)

    files = sorted(f for f in os.listdir(model_source) if f.lower().endswith(".json"))
    start = 0 if continue_ind is None else int(continue_ind)
    selected = files[start:] if num_files is None else files[start:start + int(num_files)]

    sub56_names: Optional[set[str]] = None
    if sub56:
        with open("sub56.txt", "r", encoding="utf-8") as f:
            sub56_names = {line.strip() for line in f if line.strip()}

    micro_matched_sum = 0.0
    micro_pred_denom = 0.0
    micro_gold_denom = 0.0

    macro_precisions: List[float] = []
    macro_recalls: List[float] = []
    macro_f1s: List[float] = []

    total_llm_segments = 0

    for i, fname in enumerate(selected, 1):
        if printApiLog:
            print(f"PROCESSING: {i}/{len(selected)} {fname}")

        if sub56 and sub56_names is not None and fname not in sub56_names:
            if printApiLog:
                print("SKIPPED...")
            continue

        model_path = os.path.join(model_source, fname)
        annot_path = os.path.join(annot_source, fname)
        dst_path = os.path.join(dest_dir, fname)

        if not os.path.isfile(annot_path):
            print(f"[ERROR] Missing annotation file: {fname}")
            continue

        with open(model_path, "r", encoding="utf-8") as f:
            model_data = json.load(f)
        with open(annot_path, "r", encoding="utf-8") as f:
            annot_data = json.load(f)

        for p_idx, (p_model, p_annot) in enumerate(
            zip(model_data.get("paragraphs", []), annot_data.get("paragraphs", []))
        ):
            model_list = p_model.get("model")
            if not isinstance(model_list, list) or len(model_list) != 1:
                continue

            feedback = model_list[0].get("feedback")
            if not isinstance(feedback, list) or "FEEDBACK_SEGMENTS" not in feedback:
                continue

            seg_idx = feedback.index("FEEDBACK_SEGMENTS")
            model_segs = [
                s for s in feedback[seg_idx + 1:]
                if isinstance(s, str) and len(s.strip()) >= 3
            ]
            if not model_segs:
                continue

            comments = [
                a.get("comment", "").strip()
                for a in p_annot.get("annotations", [])
                if isinstance(a, dict) and len(a.get("comment", "").strip()) >= 3
            ]
            if not comments:
                continue

            if one_GT_only and len(comments) != 1:
                continue
            if g1_GT_only and len(comments) == 1:
                continue

            total_llm_segments += len(model_segs)

            n_model = len(model_segs)
            n_annot = len(comments)

            segsim: List[Dict[str, Any]] = []
            score_matrix = np.zeros((n_model, n_annot), dtype=float)

            # Build score matrix and save all pairwise scores into segsim
            for mi, ms in enumerate(model_segs):
                for ai, ac in enumerate(comments):
                    key = (fname, p_idx, mi, ai)

                    if key in scores_lookup:
                        score = int(scores_lookup[key])

                    elif scores_lookup_g1GT is not None and key in scores_lookup_g1GT:
                        score = int(scores_lookup_g1GT[key])

                    else:
                        if strict_missing:
                            raise KeyError(
                                f"Missing score for pair {key} in both {scores_jsonl_path} "
                                f"and {g1gt_path}."
                            )
                        score = 0

                    score_matrix[mi, ai] = score

                    segsim.append({
                        "feedback": ms,
                        "comment": ac,
                        "score": score,
                    })

            # Maximum-weight 1-to-1 bipartite matching via Hungarian assignment
            # linear_sum_assignment minimizes cost, so use negative scores.
            row_ind, col_ind = linear_sum_assignment(-score_matrix)

            matched_sum = float(score_matrix[row_ind, col_ind].sum())

            pred_denom = MAX_SELF_SIM * n_model
            gold_denom = MAX_SELF_SIM * n_annot

            precision = matched_sum / pred_denom if pred_denom > 0 else 0.0
            recall = matched_sum / gold_denom if gold_denom > 0 else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if (precision + recall) > 0
                else 0.0
            )

            if printApiLog:
                matched_pairs = [
                    {
                        "model_seg_index": int(r),
                        "annot_index": int(c),
                        "score": float(score_matrix[r, c]),
                    }
                    for r, c in zip(row_ind, col_ind)
                ]
                print({
                    "file": fname,
                    "paragraph": p_idx,
                    "matched_sum": matched_sum,
                    "n_model": n_model,
                    "n_annot": n_annot,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "matched_pairs": matched_pairs,
                })

            micro_matched_sum += matched_sum
            micro_pred_denom += pred_denom
            micro_gold_denom += gold_denom

            macro_precisions.append(precision)
            macro_recalls.append(recall)
            macro_f1s.append(f1)

            p_model["segsim"] = segsim

        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(model_data, f, ensure_ascii=False, indent=2)

    micro_precision = (
        micro_matched_sum / micro_pred_denom if micro_pred_denom > 0 else 0.0
    )
    micro_recall = (
        micro_matched_sum / micro_gold_denom if micro_gold_denom > 0 else 0.0
    )
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0
        else 0.0
    )

    metrics = {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": sum(macro_precisions) / len(macro_precisions) if macro_precisions else 0.0,
        "macro_recall": sum(macro_recalls) / len(macro_recalls) if macro_recalls else 0.0,
        "macro_f1": sum(macro_f1s) / len(macro_f1s) if macro_f1s else 0.0,
        "total_llm_segments": total_llm_segments,
    }

    if printApiLog:
        print("\n=== FINAL METRICS ===")
        print(metrics)

    return metrics

In [ ]:
# Stage 2 (Assuming Stage 1 is already ran, whether GPT or Gemini, doesn't matter)

setting = 's53'

metrics = unimatch_from_saved_scores_soft_1to1(
    model_source="./segment_individual/seg_"+setting,
    annot_source="./segment_individual/seg_annot/",
    dest_dir="./unimatch_individual/soft_1to1_"+setting+'/',
    scores_jsonl_path="./unimatch_individual/gemini_buffer_stage1_scores_"+setting+".jsonl",
    num_files=None,
    continue_ind=None,
    printApiLog=False,
    sub56=False,
    one_GT_only=True,
    g1_GT_only=False,
    strict_missing=True,
)

print(metrics)